# 공정 조합 퍼널 · 실제 데이터 분석

설비 이력(`raw.csv`)에서 bad를 끌어올린 **설비 · 챔버 경로**를 찾습니다. 위에서부터 실행하면 됩니다.

**보는 순서**
1. **한 장 funnel**: 모든 STEP의 조합을 한 장에 그립니다. 기준선 밖 = 불량 판정
2. **혐의 대상 표**: 고치면 줄어드는 bad 웨이퍼(초과 bad)가 많은 순
3. **자세히**: 어느 Order · 유닛인지, 나머지 웨이퍼와 얼마나 다른지, 언제부터인지
4. **경계선 비교**: 지금 쓰는 Bonferroni 경계와 √(2 ln M_N) 경계를 같은 조합에 따로 그려 봅니다

데이터 점검 결과와 판정 방식 설명은 맨 아래 **부록**에 있습니다.

**데이터**: 웨이퍼 = `root_lot_id` + `wafer_id`(한 장이 하나의 표본) · STEP = `step_seq`에서 끝의 `_번호`를 뗀 값 · Order = 그 번호(예: `ex100000_3` → STEP `ex100000`의 Order 3, 경로에는 `O3`으로 적음) · 유닛 = `eqp_id-chamber_id` · Value = `y_value` · good_bad = `G` · `B` · `N`(N은 계산에서 빼고 차트에만 표시)

## 설정
`unit_combi` 폴더의 `raw.csv`를 읽습니다. demo로 해 보려면 `demo_raw.csv`를 `raw.csv`로 복사하세요. 값을 바꾼 뒤 위에서부터 다시 실행합니다.

In [ ]:
MIN_N = 20             # 조합에 필요한 최소 웨이퍼 수 (웨이퍼가 1,000장보다 적으면 10 정도)
MAX_DEPTH = 3          # 조합에 묶을 최대 Order 수 (늘리면 계산이 늘고 모든 기준선이 조금 올라감)
PICK = 1               # 자세히 볼 혐의 대상의 순위
STEP = None            # 특정 STEP만 볼 때 (예: 'ex100000'). None이면 전체
SAVE_RANKING = None    # 'ranking.csv'처럼 주면 혐의 대상 표(모든 열)를 CSV로 저장

In [ ]:
# 고급 설정: 보통 그대로 둡니다
PART_ID = None         # 한 제품만 볼 때 지정 (None이면 모든 part를 함께)
LINE_ID = None         # 한 라인만 볼 때 지정
STEP_SUFFIX = r'_(\d+)$'  # step_seq 끝의 Order 번호 규칙. None이면 step_seq를 STEP, step_ord를 Order로
HIGHER_IS_WORSE = True # Value가 클수록 나쁘면 True (수율처럼 클수록 좋으면 False)
Y_TRANSFORM = 'auto'   # 한쪽으로 치우친 Value: 'auto' 치우치면 순위로 · 'rank' · 'log' · None 그대로
PART_ADJUST = 'auto'   # part마다 수준이 다를 때: 'auto' 뚜렷하면 맞춤 · True 항상 · False 안 함
REWORK = 'last'        # 같은 웨이퍼 · STEP · Order 이력이 여러 줄이면: 'last' 마지막 · 'first' 처음
SAME_WAFERS = 0.9      # 웨이퍼가 이만큼 겹치는 혐의 대상은 한 줄로 묶음 (None이면 안 묶음)
SPREAD_ADJUST = True   # Order 1개 조합의 z가 이론보다 넓게 퍼져 있으면 그 배수만큼 기준선을 넓힘 (False면 안 넓힘)
FLOW_MAX_ORDERS = 12   # STEP SEQ 차트에 그릴 최대 Order 수
LOGX = True            # funnel 가로축 로그
FONT_PATH = None       # 한글 글꼴 파일(.ttf). None이면 노트북 옆 fonts 폴더의 나눔고딕

In [ ]:
import math
import re
import sys
import time
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch, PathPatch
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator

assert sys.version_info >= (3, 12), f'Python 3.12 이상이 필요합니다 (현재 {sys.version.split()[0]})'
Y_NAME = 'Value' if HIGHER_IS_WORSE else '−Value'

C = {
    'surface': '#fcfcfb', 'ink': '#0b0b0b', 'ink2': '#52514e', 'muted': '#898781',
    'grid': '#e1e0d9', 'axis': '#c3c2b7', 'band': '#ecebe6',
    'bad': '#d03b3b', 'accent': '#2a78d6', 'mid': '#f0efec',
}
# 한글 글꼴: FONT_PATH → 노트북 옆 fonts 폴더(저장소에 든 나눔고딕, SIL OFL) → 시스템 글꼴 순으로 찾는다
font_files = [Path(FONT_PATH)] if FONT_PATH else [
    p for d in (Path.cwd() / 'fonts', Path.cwd() / 'notebooks' / 'fonts') if d.is_dir() for p in sorted(d.glob('*.tt[fc]'))]
for p in font_files:
    mpl.font_manager.fontManager.addfont(str(p))
if font_files:
    KR_FONT = mpl.font_manager.FontProperties(fname=str(font_files[0])).get_name()
else:
    _fonts = {f.name for f in mpl.font_manager.fontManager.ttflist}
    KR_FONT = next((f for f in ('NanumGothic', 'Nanum Gothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'Malgun Gothic',
                                'Apple SD Gothic Neo', 'AppleGothic') if f in _fonts), None)
if KR_FONT is None:
    print('⚠ 한글 글꼴을 찾지 못해 차트의 한글이 □로 보일 수 있습니다. 저장소의 notebooks/fonts 폴더를 노트북 옆에 두거나 FONT_PATH를 지정하세요.')
plt.rcParams.update({
    'font.family': [KR_FONT or 'DejaVu Sans', 'DejaVu Sans'], 'axes.unicode_minus': False, 'font.size': 10,
    'figure.dpi': 100, 'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'], 'savefig.facecolor': C['surface'],
    'axes.edgecolor': C['axis'], 'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlecolor': C['ink'], 'axes.titlepad': 10,
    'axes.labelcolor': C['ink2'], 'axes.labelsize': 10,
    'xtick.color': C['axis'], 'ytick.color': C['axis'], 'xtick.labelcolor': C['ink2'], 'ytick.labelcolor': C['ink2'],
    'xtick.major.size': 0, 'ytick.major.size': 0, 'xtick.minor.size': 0, 'ytick.minor.size': 0,
    'legend.frameon': False, 'legend.fontsize': 9,
})
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:,.4g}'.format)

## 1. 데이터 읽기
`raw.csv`를 읽어 웨이퍼마다 STEP · Order별로 지난 유닛을 정리합니다. 확인할 점이 있으면 알려 주고, 자세한 점검 표는 부록 A에 있습니다.

In [ ]:
REQUIRED = ['analysis_date', 'job_id', 'root_lot_id', 'lot_id', 'wafer_id', 'tkin_time', 'line_id', 'part_id',
            'step_seq', 'step_desc', 'eqp_id', 'chamber_id', 'ppid', 'y_value', 'good_bad', 'step_ord']
USE = [c for c in REQUIRED if c not in ('analysis_date', 'job_id', 'lot_id', 'ppid')]   # 분석에 쓰는 컬럼만 읽는다 (raw 하나 = job 하나)
GB_CODE = {'G': 0.0, 'B': 1.0}                     # good_bad: G · B만 계산에 쓰고, N은 계산에서 빼고 차트에만 표시


def ord_label(o):
    return str(int(o)) if float(o).is_integer() else str(o)


def cat_values(s, f, fill=np.nan):
    """category 컬럼의 고유값에만 f를 적용해 줄마다의 값 배열로 (결측은 fill). 수백만 줄도 고유값 수만큼만 계산한다"""
    vals = np.asarray(f(pd.Series(s.cat.categories, dtype=object)))
    return np.concatenate([vals, np.array([fill]).astype(vals.dtype)])[s.cat.codes.to_numpy()]


def cat_recode(s, f):
    """category 컬럼의 고유값에 f를 적용한 새 category (바꾼 값이 같아지면 하나로 합친다)"""
    new = np.asarray(f(pd.Series(s.cat.categories, dtype=object)), dtype=object)
    ok = pd.notna(new)
    u, inv = np.unique(new[ok].astype(str), return_inverse=True)
    code = np.full(len(new) + 1, -1)
    code[np.flatnonzero(ok)] = inv
    return pd.Categorical.from_codes(code[s.cat.codes.to_numpy()], u)


def norm_wid(v):
    """wafer_id가 숫자면 '01'과 '1'을 같게 본다"""
    num = pd.to_numeric(v, errors='coerce')
    is_int = num.notna() & (num == num.round())
    out = v.copy()
    out[is_int] = num[is_int].astype(np.int64).astype(str)
    return out


t0 = time.perf_counter()
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = REPO / 'raw.csv'               # unit_combi 폴더의 raw.csv 하나만 읽는다
assert CSV_PATH.exists(), (f'raw.csv가 없습니다: {CSV_PATH}\n실제 데이터를 이 이름으로 두세요. '
                           f'demo로 확인하려면 같은 폴더의 demo_raw.csv를 raw.csv로 복사하세요.')
miss_cols = [c for c in REQUIRED if c not in pd.read_csv(CSV_PATH, nrows=0).columns]
assert not miss_cols, f'CSV에 없는 컬럼: {miss_cols}'
# 컬럼마다 값 종류가 줄 수보다 훨씬 적어서 category로 읽으면 메모리가 약 10분의 1로 줄고, 공백 정리 · 변환도 고유값에만 하면 된다.
# tkin_time은 줄마다 거의 달라서 문자열로 읽어 바로 시각으로 바꾼다
raw = pd.read_csv(CSV_PATH, usecols=USE, dtype={**{c: 'category' for c in USE}, 'tkin_time': str})
raw['t'] = pd.to_datetime(raw.pop('tkin_time'), errors='coerce')
for c in USE:
    if c != 'tkin_time':
        raw[c] = cat_recode(raw[c], lambda v: v.str.strip())
n_read = len(raw)

# 웨이퍼 키: root_lot_id와 wafer_id 코드의 쌍 (정수)
wid = cat_recode(raw['wafer_id'], norm_wid)
root_code, wid_code = raw['root_lot_id'].cat.codes.to_numpy().astype(np.int64), wid.codes.astype(np.int64)
no_id = (root_code < 0) | (wid_code < 0)
raw['wafer_key'] = root_code * (len(wid.categories) + 1) + wid_code
n_no_id = int(no_id.sum())
raw = raw[~no_id]

for col, val in (('part_id', PART_ID), ('line_id', LINE_ID)):
    if val is not None:
        raw = raw[raw[col] == val]

to_num = lambda v: pd.to_numeric(v, errors='coerce').to_numpy(float)
raw['y'] = cat_values(raw['y_value'], to_num)
# 스텝과 오더: step_seq 끝의 '_번호'를 떼어 스텝(step_num)으로 묶고 그 번호를 오더로 쓴다. 끝 번호가 없으면 step_seq 그대로 · step_ord
if STEP_SUFFIX:
    sfx = cat_values(raw['step_seq'], lambda v: to_num(v.str.extract(STEP_SUFFIX, expand=False)))
    raw['step'] = cat_recode(raw['step_seq'], lambda v: v.str.replace(STEP_SUFFIX, '', regex=True))
else:
    sfx = np.full(len(raw), np.nan)
    raw['step'] = raw['step_seq']
step_ord = cat_values(raw['step_ord'], to_num)
raw['ord'] = np.where(np.isnan(sfx), step_ord, sfx)
n_ord_diff = int((~np.isnan(sfx) & ~np.isnan(step_ord) & (sfx != step_ord)).sum())
# good_bad: G = 0, B = 1. N(과 G · B · N이 아닌 값)은 계산에서 빼고 차트에만 표시한다
raw['gb'] = cat_recode(raw['good_bad'], lambda v: v.str.upper())
raw['bad'] = cat_values(raw['gb'], lambda v: v.map(GB_CODE).to_numpy(float))
gb_other = raw['gb'].value_counts(dropna=False)
gb_other = gb_other[[k not in ('G', 'B', 'N') for k in gb_other.index] & (gb_other > 0).to_numpy()]

# 웨이퍼 표: 웨이퍼마다 값 하나
wf = raw.groupby('wafer_key').agg(root_lot_id=('root_lot_id', 'first'), part=('part_id', 'first'), part_n=('part_id', 'nunique'),
                                  y=('y', 'first'), y_n=('y', 'nunique'), bad=('bad', 'first'), gb=('gb', 'first'), gb_n=('gb', 'nunique'))
wf['root_lot_id'], wf['part'] = wf['root_lot_id'].astype(object), wf['part'].astype(object)
assert wf['y'].notna().all(), f"y_value가 비어 있거나 숫자가 아닌 웨이퍼가 {int(wf['y'].isna().sum()):,}장 있습니다. 입력을 확인하세요."
has_bad = bool(wf['bad'].notna().any())                 # G · B가 하나도 없으면 y_value만으로 판정 (이때는 모든 웨이퍼 사용)
keep = wf['bad'].notna() if has_bad else pd.Series(True, index=wf.index)   # 계산에 쓰는 웨이퍼 (G · B)

# 이력 표: 오더 · 설비가 있는 줄만. N 웨이퍼의 이력은 차트 표시용으로 따로 둔다
h = raw.loc[raw['ord'].notna() & raw['eqp_id'].notna() & raw['step'].notna(),
            ['wafer_key', 'step', 'step_seq', 'step_desc', 'ord', 'eqp_id', 'chamber_id', 't']].copy()
# 유닛 = eqp_id-chamber_id (chamber가 없으면 eqp_id). 설비 · 챔버 코드 쌍에서 고유한 쌍에만 이름을 붙인다
e_cat, c_cat = h['eqp_id'].cat.categories, h['chamber_id'].cat.categories
pair = h['eqp_id'].cat.codes.to_numpy().astype(np.int64) * (len(c_cat) + 1) + h['chamber_id'].cat.codes.to_numpy() + 1
pairs, inv = np.unique(pair, return_inverse=True)
names = np.array([e_cat[p // (len(c_cat) + 1)] + ('-' + c_cat[p % (len(c_cat) + 1) - 1] if p % (len(c_cat) + 1) and c_cat[p % (len(c_cat) + 1) - 1] else '')
                  for p in pairs], dtype=object)
u_names, u_inv = np.unique(names, return_inverse=True)
h['unit'] = pd.Categorical.from_codes(u_inv[inv], u_names)
n_dup_keys = int(h.duplicated(['wafer_key', 'step', 'ord'], keep='first').sum())
h = (h.sort_values('t', na_position='first' if REWORK == 'last' else 'last', kind='stable')
      .drop_duplicates(['wafer_key', 'step', 'ord'], keep=REWORK))
h_n = h[h['wafer_key'].isin(wf.index[~keep])]           # N 웨이퍼 이력 (4절 차트에 빈 원으로만 표시)
h = h[h['wafer_key'].isin(wf.index[keep])]
n_value = wf.loc[~keep, 'y']                            # N 웨이퍼의 Value (원래 값)
n_gb_n = int((~keep & (wf['gb'] == 'N')).sum())        # 계산에서 뺀 웨이퍼 중 N · 그 밖의 값
n_gb_other = int((~keep).sum()) - n_gb_n

INFO_ROWS = ['읽은 줄', '필터 후 줄', '웨이퍼 (root_lot_id + wafer_id)', 'good_bad가 N인 웨이퍼 (계산에서 빼고 차트에만)', '분석에 쓰는 이력 줄']
quality = pd.DataFrame([
    ('읽은 줄', n_read),
    ('root_lot_id · wafer_id가 없어 뺀 줄', n_no_id),
    ('필터 후 줄', len(raw)),
    ('웨이퍼 (root_lot_id + wafer_id)', len(wf)),
    ('good_bad가 N인 웨이퍼 (계산에서 빼고 차트에만)', n_gb_n),
    ('good_bad가 G · B · N이 아닌 웨이퍼 (N처럼 뺌)', n_gb_other),
    ('y_value가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['y_n'] > 1).sum())),
    ('good_bad가 둘 이상인 웨이퍼 (G · B 중 첫 값 사용)', int((wf['gb_n'] > 1).sum())),
    ('part_id가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['part_n'] > 1).sum())),
    ('Order 번호 · eqp_id · step_seq가 없어 뺀 줄', int((raw['ord'].isna() | raw['eqp_id'].isna() | raw['step'].isna()).sum())),
    ('step_ord가 step_seq 끝 번호와 다른 줄 (끝 번호 사용)', n_ord_diff),
    ('tkin_time을 읽지 못한 줄', int(raw['t'].isna().sum())),
    (f"같은 웨이퍼 · STEP · Order가 겹친 줄 ('{REWORK}'만 남김)", n_dup_keys),
    ('분석에 쓰는 이력 줄', len(h)),
], columns=['항목', '값']).set_index('항목')
if len(gb_other) and has_bad:
    print('⚠ good_bad에 G · B · N이 아닌 값이 있어 N처럼 계산에서 뺍니다:',
          {(k if isinstance(k, str) else '(빈 값)'): int(v) for k, v in gb_other.head(10).items()})
if n_ord_diff:
    print(f'⚠ step_ord가 step_seq 끝 번호와 다른 줄이 {n_ord_diff:,}개 있습니다. Order는 step_seq 끝 번호를 씁니다.')

# 스텝으로 묶인 모습: 오더가 많은 스텝부터
so = h.drop_duplicates(['step', 'ord']).sort_values('ord')
step_map = so.groupby('step').agg(
    오더=('ord', 'size'),
    오더_번호=('ord', lambda s: ', '.join(map(ord_label, s)) if len(s) <= 8 else f'{ord_label(s.iat[0])} … {ord_label(s.iat[-1])}'),
    step_seq=('step_seq', lambda s: s.iat[0] if len(s) == 1 else f'{s.iat[0]} … {s.iat[-1]}'),
    step_desc=('step_desc', lambda s: ' / '.join(list(dict.fromkeys(s.dropna()))[:3]) + (' …' if s.nunique() > 3 else '')))
step_map = step_map.sort_values('오더', ascending=False, kind='stable').rename(columns={'오더': 'Order 수', '오더_번호': 'Order 번호', 'step_seq': 'STEP SEQ'})

In [ ]:
MISSING = 0xFFFF


def step_sort_key(s):
    return (0, float(s), s) if s.replace('.', '', 1).isdigit() else (1, 0.0, s)


def group_mode(g, codes, cats, G):
    """묶음 g(0..G-1)마다 가장 흔한 category 값 (개수가 같으면 이름순으로 앞선 값)"""
    out = np.full(G, '', dtype=object)
    ok = codes >= 0
    if ok.any():
        pk, cnt = np.unique(g[ok] * len(cats) + codes[ok], return_counts=True)
        gg, cc = pk // len(cats), pk % len(cats)
        o = np.lexsort((cc, -cnt, gg))
        first = o[np.r_[True, gg[o][1:] != gg[o][:-1]]]
        out[gg[first]] = np.asarray(cats, dtype=object)[cc[first]]
    return out


t0 = time.perf_counter()
wafers = wf.index[keep]
N = len(wafers)
# 줄마다 (스텝 순위, 오더, 유닛)으로 한 번 정렬하고 경계만 찾아서, 스텝 · 오더마다 따로 거르지 않고 배열을 채운다
code_of = lambda col: h[col].cat.codes.to_numpy().astype(np.int64)
step_names = np.asarray(h['step'].cat.categories, dtype=object)
order_ = sorted(np.unique(code_of('step')), key=lambda c: step_sort_key(step_names[c]))
rank = np.full(len(step_names), -1)
rank[order_] = np.arange(len(order_))
s_arr, o_arr, u_arr = rank[code_of('step')], h['ord'].to_numpy(float), code_of('unit')
w_arr = pd.Series(np.arange(N), index=wafers).reindex(h['wafer_key'].to_numpy()).to_numpy().astype(np.int64)
srt = np.lexsort((u_arr, o_arr, s_arr))
s_arr, o_arr, u_arr, w_arr = s_arr[srt], o_arr[srt], u_arr[srt], w_arr[srt]
t_arr = h['t'].to_numpy('datetime64[ns]')[srt]
new_g = np.r_[True, (s_arr[1:] != s_arr[:-1]) | (o_arr[1:] != o_arr[:-1])]
g_arr = np.cumsum(new_g) - 1                            # (스텝, 오더) 묶음 번호
G, g_start = int(g_arr[-1]) + 1, np.flatnonzero(new_g)
new_pu = new_g | np.r_[True, u_arr[1:] != u_arr[:-1]]
pu = np.cumsum(new_pu) - 1                              # (스텝, 오더, 유닛) 고유 쌍 번호
local = pu - pu[g_start][g_arr]                         # 오더 안 유닛 번호 (유닛 이름순)
assert local.max() < MISSING, '한 Order의 유닛이 너무 많습니다'
pu_start = np.flatnonzero(new_pu)
units_of = np.split(np.asarray(h['unit'].cat.categories, dtype=object)[u_arr[pu_start]], np.flatnonzero(np.diff(g_arr[pu_start])) + 1)
seq_mode = group_mode(g_arr, code_of('step_seq')[srt], h['step_seq'].cat.categories, G)
desc_mode = group_mode(g_arr, code_of('step_desc')[srt], h['step_desc'].cat.categories, G)
proc_mode = group_mode(s_arr, code_of('step_desc')[srt], h['step_desc'].cat.categories, len(order_))
t_int = np.where(np.isnat(t_arr), np.iinfo(np.int64).max, t_arr.view(np.int64))
bounds = np.r_[np.flatnonzero(np.r_[True, s_arr[1:] != s_arr[:-1]]), len(s_arr)]
steps, assign, trackin = [], [], []
for s_i, (r0, r1) in enumerate(zip(bounds[:-1], bounds[1:])):
    q0, q1 = g_arr[r0], g_arr[r1 - 1] + 1
    A = np.full((q1 - q0, N), MISSING, dtype=np.uint16)
    A[g_arr[r0:r1] - q0, w_arr[r0:r1]] = local[r0:r1]
    first_t = np.full(N, np.iinfo(np.int64).max)
    np.minimum.at(first_t, w_arr[r0:r1], t_int[r0:r1])   # 웨이퍼마다 이 스텝의 첫 track-in
    tk = first_t.view('datetime64[ns]').copy()
    tk[first_t == np.iinfo(np.int64).max] = np.datetime64('NaT', 'ns')
    seqs = [{'name': f'O{ord_label(o_arr[g_start[g]])}', 'ord': float(o_arr[g_start[g]]), 'units': list(units_of[g]),
             'seq': seq_mode[g], 'desc': desc_mode[g]} for g in range(q0, q1)]
    steps.append({'index': s_i, 'name': str(step_names[order_[s_i]]), 'proc': proc_mode[s_i], 'seqs': seqs})
    assign.append(A)
    trackin.append(tk)

Y = wf.loc[wafers, 'y'].to_numpy(float)
if not HIGHER_IS_WORSE:
    Y = -Y                                     # 작을수록 나쁜 값은 부호를 뒤집어 '클수록 나쁨'으로 맞춘다
assert N >= 2 * MIN_N and Y.std() > 0, f'비교할 수 없습니다: 웨이퍼 {N}장, Value 표준편차 {Y.std():.3g}'
B = wf.loc[wafers, 'bad'].to_numpy(float) if has_bad else None
if B is None:
    print('⚠ good_bad에 G · B가 없어 Value만으로 판정합니다 (모든 웨이퍼 사용).')
elif B.std() == 0:
    print(f'⚠ good_bad가 한쪽뿐입니다 (bad 비율 {B.mean():.0%}). 양품 · 불량 웨이퍼가 함께 있어야 비교할 수 있어서 Value만으로 판정합니다.')
    B = None
lots = wf.loc[wafers, 'root_lot_id'].to_numpy()
lot_idx, lot_names = pd.factorize(lots)
data = {'N': N, 'Y': Y, 'B': B, 'steps': steps, 'assign': assign, 'trackin': trackin, 'wafers': wafers,
        'lots': lots, 'lot_idx': lot_idx, 'n_lots': len(lot_names), 'parts': wf.loc[wafers, 'part'].fillna('').to_numpy(),
        'bad': wf.loc[wafers, 'bad'].to_numpy(float),
        'value': wf.loc[wafers, 'y'].to_numpy(float)}
n_orders = sum(len(s['seqs']) for s in steps)
issues = int((quality['값'].drop(INFO_ROWS) > 0).sum())
print(f"데이터: 웨이퍼 {N:,}장" + (f" (good_bad N {n_gb_n:,}장" + (f" · 그 밖의 값 {n_gb_other:,}장" if n_gb_other else '')
                                  + '은 계산에서 빼고 차트에만 표시)' if len(n_value) else '')
      + f" · 랏 {data['n_lots']:,}개 · STEP {len(steps):,}개 (STEP SEQ {so['step_seq'].nunique():,}개)"
      + (f" · bad 비율 {np.nanmean(data['bad']):.1%}" if B is not None else '')
      + (f" · 확인할 점 {issues}건 (부록 A)" if issues else ' · 이상 없음'))

## 계산 함수
판정 · 표 · 차트에 쓰는 함수입니다. 수정할 필요 없이 실행만 하면 됩니다. 계산 방식은 부록 C에 있습니다.

In [ ]:
Z95, FWER_ALPHA = 1.959964, 0.05
SQRT2 = math.sqrt(2)
_A = (-39.69683028665376, 220.9460984245205, -275.9285104469687, 138.357751867269, -30.66479806614716, 2.506628277459239)
_B = (-54.47609879822406, 161.5858368580409, -155.6989798598866, 66.80131188771972, -13.28068155288572)
_C = (-0.007784894002430293, -0.3223964580411365, -2.400758277161838, -2.549732539343734, 4.374664141464968, 2.938163982698783)
_D = (0.007784695709041462, 0.3224671290700398, 2.445134137142996, 3.754408661907416)
_LG = (76.18009172947146, -86.50532032941677, 24.01409824083091, -1.231739572450155, 0.1208650973866179e-2, -0.5395239384953e-5)


def erfc(x):
    x = np.asarray(x, dtype=float)
    z = np.abs(x)
    t = 1 / (1 + 0.5 * z)
    poly = 1.00002368 + t * (0.37409196 + t * (0.09678418 + t * (-0.18628806 + t * (0.27886807 + t * (-1.13520398 + t * (1.48851587 + t * (-0.82215223 + t * 0.17087277)))))))
    r = t * np.exp(-z * z - 1.26551223 + t * poly)
    return np.where(x >= 0, r, 2 - r)


def norm_cdf(z):
    return 0.5 * erfc(-np.asarray(z, dtype=float) / SQRT2)


def norm_sf(z):
    return 0.5 * erfc(np.asarray(z, dtype=float) / SQRT2)


def norm_inv(p):
    if p <= 0:
        return -math.inf
    if p >= 1:
        return math.inf
    if p < 0.02425:
        q = math.sqrt(-2 * math.log(p))
        x = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    elif p <= 1 - 0.02425:
        q = p - 0.5
        r = q * q
        x = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    else:
        q = math.sqrt(-2 * math.log(1 - p))
        x = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    e = float(norm_cdf(x)) - p
    u = e * math.sqrt(2 * math.pi) * math.exp((x * x) / 2)
    return x - u / (1 + (x * u) / 2)


def norm_isf(p):
    """상단 꼬리확률 → z (배열, 아주 작은 p도 정밀하게)"""
    p = np.clip(np.atleast_1d(np.asarray(p, dtype=float)), 1e-300, 1 - 1e-16)
    x = np.empty_like(p)
    lo, hi = p < 0.02425, p > 1 - 0.02425
    mid = ~(lo | hi)
    q = np.sqrt(-2 * np.log(p[lo]))
    x[lo] = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    q = p[mid] - 0.5
    r = q * q
    x[mid] = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    q = np.sqrt(-2 * np.log(1 - p[hi]))
    x[hi] = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    with np.errstate(all='ignore'):
        e = norm_cdf(x) - p
        u = e * math.sqrt(2 * math.pi) * np.exp((x * x) / 2)
        refined = x - u / (1 + (x * u) / 2)
    return -np.where(np.isfinite(refined) & (p > 1e-250), refined, x)


def lgamma(x):
    x = np.asarray(x, dtype=float)
    y = x.copy()
    tmp = x + 5.5 - (x + 0.5) * np.log(x + 5.5)
    ser = 1.000000000190015
    for g in _LG:
        y = y + 1
        ser = ser + g / y
    return -tmp + np.log((2.5066282746310005 * ser) / x)


def betacf(a, b, x):
    a, b, x = np.broadcast_arrays(*(np.asarray(v, dtype=float) for v in (a, b, x)))
    MAXIT, EPS, FPMIN = 300, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1, a - 1
    floor = lambda v: np.where(np.abs(v) < FPMIN, FPMIN, v)
    c = np.ones_like(x)
    d = 1 / floor(1 - (qab * x) / qap)
    h_ = d.copy()
    live = np.ones(x.shape, dtype=bool)
    with np.errstate(all='ignore'):
        for m in range(1, MAXIT + 1):
            m2 = 2 * m
            aa = (m * (b - m) * x) / ((qam + m2) * (a + m2))
            d1 = 1 / floor(1 + aa * d)
            c1 = floor(1 + aa / c)
            h1 = h_ * (d1 * c1)
            aa = (-(a + m) * (qab + m) * x) / ((a + m2) * (qap + m2))
            d1 = 1 / floor(1 + aa * d1)
            c1 = floor(1 + aa / c1)
            de = d1 * c1
            h1 = h1 * de
            c, d, h_ = np.where(live, c1, c), np.where(live, d1, d), np.where(live, h1, h_)
            live &= ~(np.abs(de - 1) < EPS)
            if not live.any():
                break
    return h_


def ibeta(x, a, b):
    x, a, b = np.broadcast_arrays(*(np.atleast_1d(np.asarray(v, dtype=float)) for v in (x, a, b)))
    out = np.where(x <= 0, 0.0, 1.0)
    mid = (x > 0) & (x < 1)
    if mid.any():
        xm, am, bm = x[mid], a[mid], b[mid]
        bt = np.exp(lgamma(am + bm) - lgamma(am) - lgamma(bm) + am * np.log(xm) + bm * np.log(1 - xm))
        front = xm < (am + 1) / (am + bm + 2)
        r = np.empty_like(xm)
        if front.any():
            r[front] = (bt[front] * betacf(am[front], bm[front], xm[front])) / am[front]
        if (~front).any():
            r[~front] = 1 - (bt[~front] * betacf(bm[~front], am[~front], 1 - xm[~front])) / bm[~front]
        out[mid] = r
    return out


def t_sf(t, df):
    t, df = np.atleast_1d(np.asarray(t, dtype=float)), np.atleast_1d(np.asarray(df, dtype=float))
    p = 0.5 * ibeta(df / (df + t * t), df / 2, 0.5)
    return np.where(t >= 0, p, 1 - p)


def f_sf(f, d1, d2):
    f = np.atleast_1d(np.asarray(f, dtype=float))
    with np.errstate(all='ignore'):
        p = ibeta(d2 / (d2 + d1 * f), d2 / 2, d1 / 2)
    return np.where(f <= 0, 1.0, p)


def var_of(n, s, ss):
    return max(0.0, (ss - (s * s) / n) / (n - 1)) if n > 1 else 0.0


def anova(n, s, ss):
    keep_ = n > 0
    n, s, ss = n[keep_].tolist(), s[keep_].tolist(), ss[keep_].tolist()
    k, N_ = len(n), sum(n)
    if k < 2 or N_ - k < 1:
        return None
    grand = sum(s) / N_
    ssb = sum(ni * (si / ni - grand) ** 2 for ni, si in zip(n, s))
    ssw = sum(ssi - ni * (si / ni) ** 2 for ni, si, ssi in zip(n, s, ss))
    if ssw <= 0:
        return None
    d1, d2 = k - 1, N_ - k
    F = ssb / d1 / (ssw / d2)
    return {'F': F, 'd1': d1, 'd2': d2, 'p': float(f_sf(F, d1, d2)[0])}


def welch(na, sa, ssa, nb, sb, ssb):
    """a가 b보다 큰지 단측 Welch t (요약통계 배열)"""
    na, sa, ssa, nb, sb, ssb = (np.atleast_1d(np.asarray(v, dtype=float)) for v in (na, sa, ssa, nb, sb, ssb))
    t, df, p = np.zeros_like(na), np.ones_like(na), np.ones_like(na)
    ok = (na >= 2) & (nb >= 2)
    with np.errstate(all='ignore'):
        va = np.maximum(0, (ssa - (sa * sa) / na) / (na - 1)) / na
        vb = np.maximum(0, (ssb - (sb * sb) / nb) / (nb - 1)) / nb
        se = np.sqrt(va + vb)
        ok &= se != 0
        t[ok] = (sa[ok] / na[ok] - sb[ok] / nb[ok]) / se[ok]
        df[ok] = (va[ok] + vb[ok]) ** 2 / ((va[ok] * va[ok]) / (na[ok] - 1) + (vb[ok] * vb[ok]) / (nb[ok] - 1))
    if ok.any():
        p[ok] = t_sf(t[ok], df[ok])
    return t, df, p


def quantile(v, p):
    if not len(v):
        return math.nan
    h_ = (len(v) - 1) * p
    lo = math.floor(h_)
    return float(v[lo] + (h_ - lo) * (v[min(lo + 1, len(v) - 1)] - v[lo]))

In [ ]:
def combo_key(s, items):
    return f'{s}|' + '|'.join(f'{q}:{u}' for q, u in items)


def binom_fix(n, x, p0):
    """bad(0/1) z의 꼬리 보정 배수. bad 비율이 낮고 웨이퍼가 적으면 bad 장수 분포가 오른쪽으로 길어서
    정규 근사 z가 실제보다 커진다. 정확한 이항 꼬리확률 P(bad ≥ x)로 구한 z ÷ 정규 근사 z를 곱해 줄인다(1보다 크게 하지는 않음)"""
    n, x = np.asarray(n, dtype=float), np.rint(np.asarray(x, dtype=float))
    z_norm = (x - n * p0) / np.sqrt(n * p0 * (1 - p0))
    fix = np.ones_like(n)
    up = z_norm > 0.5
    if up.any() and 0 < p0 < 1:
        nu, xu = n[up], x[up]
        ge = lambda k: np.where(k <= 0, 1.0, ibeta(np.full_like(nu, p0), np.maximum(k, 1), np.maximum(nu - k + 1, 1e-9)))
        fix[up] = np.clip(np.minimum(norm_isf(np.maximum(ge(xu), 1e-300)), 38.0) / z_norm[up], 0.0, 1.0)
    return fix


def shrink(n, N):
    """√(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수). 전체 평균에는 조합의 웨이퍼도 들어 있어서, 조합 평균이 전체 평균에서
    우연히 벗어나는 폭은 σ/√n보다 이만큼 좁다. z를 이 값으로 나누면 조합을 나머지 웨이퍼와 비교한 z가 된다"""
    return np.sqrt(np.clip(1 - np.asarray(n, dtype=float) / N, 1 / N, None))


def analyze(data, min_n=20, max_depth=3, full_depth=2):
    """조합 생성과 조합별 지표 (웹 js/analysis.js analyze와 같은 조합 · 관문 로직).
    웨이퍼(root_lot_id + wafer_id) 한 장을 하나의 표본으로 보고, 신호(Value, bad)마다 조합을 나머지 웨이퍼와 비교한
    z = (조합 평균 − 전체 평균) ÷ (표준편차 ÷ √웨이퍼 수) ÷ √(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수)를 구해 결합 z = (z_Value + z_bad) ÷ √(2 + 2r)로 판정한다.
    bad z는 정확한 이항 꼬리확률로 맞추고, r은 두 z가 우연히 같이 움직이는 정도(Order 1개 조합에서 잰 상관, 0~0.95)라
    같은 정보를 두 번 세지 않게 한다. good_bad가 없으면 z_Value만 쓴다"""
    N, steps, assign = data['N'], data['steps'], data['assign']
    sig = {'y': data['Y']} | ({'b': data['B']} if data.get('B') is not None else {})
    base = {k_: (float(v.mean()), float(v.std(ddof=1))) for k_, v in sig.items()}
    Y = sig['y']
    Y2 = Y * Y
    mu, sd = base['y']
    bad01 = np.nan_to_num(data['bad']) if 'b' in sig else None
    p0 = float(bad01.mean()) if 'b' in sig else 0.0

    def cell_keys(s, qs):
        A = assign[s][list(qs)]
        dims = [len(steps[s]['seqs'][q]['units']) for q in qs]
        ok = np.all(A != MISSING, axis=0)
        key = np.zeros(N, dtype=np.int64)
        for i, d in enumerate(dims):
            key = key * d + A[i]
        return dims, ok, key

    def cells(s, qs):
        """오더 묶음 qs에서 웨이퍼가 실제로 지난 칸(유닛 조합)만: 칸 번호(오름차순) · 칸마다 웨이퍼 수 · 웨이퍼 → 칸 순번.
        가능한 칸 수(유닛 수의 곱)가 아무리 커도 웨이퍼 수만큼만 계산한다"""
        dims, ok, key = cell_keys(s, qs)
        u, inv, cnt = np.unique(key[ok], return_inverse=True, return_counts=True)
        return dims, ok, u, inv, cnt

    def stats(ok, inv, m, idx):
        """칸마다 신호별 합을 구해 idx 칸만 돌려준다"""
        w = lambda v: np.bincount(inv, weights=v[ok], minlength=m)[idx]
        out = {'ss': w(Y2)} | {f'sum_{k_}': w(v) for k_, v in sig.items()}
        if 'b' in sig:
            out['x_bad'] = w(bad01)                   # 원래 bad 장수 (0/1, part 보정 전): 이항분포 꼬리 보정용
        return out

    def sig_z(k_, nn, col):
        """조합들의 신호 k_ z: (조합 평균 − 전체 평균) ÷ (표준편차 ÷ √웨이퍼 수) ÷ √(1 − 웨이퍼 수 ÷ 전체 웨이퍼 수)"""
        m_, s_ = base[k_]
        return (col(f'sum_{k_}') / nn - m_) / (s_ / np.sqrt(nn)) / shrink(nn, N)

    def decode(qs, dims, key):
        units = []
        for d in reversed(dims):
            units.append(key % d)
            key //= d
        return tuple((q, int(u)) for q, u in zip(qs, reversed(units)))

    def parent_keys(dims, keys, i):
        """칸 번호에서 i번째 오더 자리를 뺀 부모 칸 번호"""
        low = math.prod(dims[i + 1:])
        return (keys // (low * dims[i])) * low + keys % low

    seq_info = []
    for s, st in enumerate(steps):
        rows = []
        for q, sq in enumerate(st['seqs']):
            a = assign[s][q]
            ok, nu = a != MISSING, len(sq['units'])
            n_u = np.bincount(a[ok], minlength=nu)
            s_u, ss_u = np.bincount(a[ok], weights=Y[ok], minlength=nu), np.bincount(a[ok], weights=Y2[ok], minlength=nu)
            units = [{'name': name, 'n': int(n_u[u]), 'mean': s_u[u] / n_u[u] if n_u[u] else math.nan,
                      'sd': math.sqrt(var_of(int(n_u[u]), s_u[u], ss_u[u]))} for u, name in enumerate(sq['units'])]
            testable = nu >= 2
            rows.append({'name': sq['name'], 'units': units, 'testable': testable, 'anova': anova(n_u, s_u, ss_u) if testable else None})
        seq_info.append(rows)

    combos, by_key, r = [], {}, 0.0
    flagged = {}                        # (스텝, 오더 묶음) → z 1.96을 넘은 칸 번호들 (다음 깊이의 관문)
    has_big = set()                     # 웨이퍼 MIN_N장 이상인 칸이 있는 (스텝, 오더 묶음). 부모에 없으면 자식에도 없다
    multi = [[q for q, sq in enumerate(st['seqs']) if len(sq['units']) >= 2] for st in steps]
    search = [0] * (max_depth + 1)
    for k in range(1, max_depth + 1):
        gated = k > full_depth
        cand = []
        for s in range(len(steps)):
            if len(multi[s]) < k:
                continue
            for qs in combinations(multi[s], k):
                subs = [(s, qs[:i] + qs[i + 1:]) for i in range(k)]
                if k > 1 and not all(p in has_big for p in subs):
                    continue                    # 오더 하나를 뺀 묶음에 큰 칸이 없으면 이 묶음에도 없다
                dims, ok, u, inv, cnt = cells(s, qs)
                big = np.flatnonzero(cnt >= min_n)
                search[k] += len(big)           # 기준 z의 보정에는 계산을 건너뛴 칸도 센다
                if len(big):
                    has_big.add((s, qs))
                if not len(big) or (gated and not any(p in flagged for p in subs)):
                    continue
                if gated:                       # 오더 하나를 뺀 부모 조합 중 하나라도 z 1.96을 넘은 칸만 계산
                    keep = np.zeros(len(big), dtype=bool)
                    for i, p in enumerate(subs):
                        if p in flagged:
                            keep |= np.isin(parent_keys(dims, u[big], i), flagged[p])
                    big = big[keep]
                    if not len(big):
                        continue
                cand.append((s, qs, dims, u[big], cnt[big], stats(ok, inv, len(u), big)))
        if not cand:
            continue
        col = lambda f: np.concatenate([c[5][f] for c in cand]).astype(float)
        nn = np.concatenate([c[4] for c in cand]).astype(float)
        sm, ss = col('sum_y'), col('ss')
        zy = sig_z('y', nn, col)
        zb = sig_z('b', nn, col) * binom_fix(nn, col('x_bad'), p0) if 'b' in sig else None
        if zb is not None and k == 1 and len(zy) > 2:   # 두 z의 상관: 오더 1개 조합(대부분 우연)에서 잰다
            r = float(np.clip(np.corrcoef(zy, zb)[0, 1], 0.0, 0.95))
        z = zy if zb is None else (zy + zb) / math.sqrt(2 + 2 * r)
        bad_rate = col('x_bad') / nn if 'b' in sig else None   # 원래(0/1) bad 비율 — 랭킹의 초과 bad 웨이퍼 계산용
        j = 0
        for s, qs, dims, keys, _, _ in cand:
            hot = []
            for key in keys.tolist():
                items = decode(qs, dims, key)
                c = {'id': len(combos), 'key': combo_key(s, items), 'step': s, 'k': k, 'items': items,
                     'n': int(nn[j]), 'sum': float(sm[j]), 'ss': float(ss[j]), 'mean': float(sm[j] / nn[j]),
                     'z': float(z[j]), 'z_y': float(zy[j]), 'z_b': float(zb[j]) if zb is not None else math.nan,
                     'bad_rate': float(bad_rate[j]) if bad_rate is not None else math.nan}
                combos.append(c)
                by_key[c['key']] = c
                if c['z'] > Z95:
                    hot.append(key)
                j += 1
            if hot:
                flagged[(s, qs)] = np.array(hot, dtype=np.int64)
    depth_count = [0] * (max_depth + 1)
    for c in combos:
        depth_count[c['k']] += 1
    return {'mu': mu, 'sd': sd, 'N': N, 'max_depth': max_depth, 'signals': list(sig), 'r': r, 'p0': p0 if 'b' in sig else None,
            'seq_info': seq_info, 'combos': combos, 'by_key': by_key, 'depth_count': depth_count, 'search_space': search}


def thresholds(res):
    """오더 수별 기준 z (Bonferroni): 5%를 조합이 있는 오더 수 D로 나누고, 오더 수마다 가능한 조합 수 M만큼 다시 나눈다.
    그래서 차트 전체에서 하나라도 우연히 넘을 확률이 5%다. 조합이 없는 오더 수는 NaN.
    퍼짐 보정: 오더 1개 조합의 z는 우연이면 퍼짐이 1이어야 한다. 같은 랏 웨이퍼가 함께 움직이거나 시간에 따라 값이 변하면
    더 넓게 퍼지므로, 그 배수(가운데 50% 폭 ÷ 1.349, 1보다 작으면 1)만큼 기준을 넓힌다(SPREAD_ADJUST)"""
    D = sum(1 for m in res['search_space'][1:] if m)
    z1 = np.array([c['z'] for c in res['combos'] if c['k'] == 1])
    q1, q3 = np.quantile(z1, [0.25, 0.75]) if len(z1) >= 20 else (0.0, 1.349)
    res['spread_measured'] = max(1.0, float(q3 - q1) / 1.349)
    res['spread'] = res['spread_measured'] if SPREAD_ADJUST else 1.0
    return [0.0] + [res['spread'] * norm_inv(1 - FWER_ALPHA / D / m) if m else math.nan for m in res['search_space'][1:]]


def members_of(d, c):
    """조합의 웨이퍼 (불리언 배열)"""
    A = d['assign'][c['step']]
    m = np.ones(d['N'], dtype=bool)
    for q, u in c['items']:
        m &= A[q] == u
    return m


def sig_values(d, k_):
    """신호 값: 'y' = 분석용 Value, 'b' = bad(0/1, part 보정했으면 보정값)"""
    return d['Y'] if k_ == 'y' else d['B']


def sig_mean_z(d, r, k_, m):
    """웨이퍼 묶음 m에서 신호 k_의 평균이 전체보다 높은지의 z (조합의 z와 같은 계산)"""
    v, n = sig_values(d, k_), int(m.sum())
    return float((v[m].mean() - v.mean()) / (v.std(ddof=1) / math.sqrt(n)) / shrink(n, len(v)))


def sig_diff_z(d, r, k_, a, b):
    """두 웨이퍼 묶음에서 신호 k_의 평균 차이(a − b)의 z (Welch t)"""
    v = sig_values(d, k_)
    var = v[a].var(ddof=1) / a.sum() + v[b].var(ddof=1) / b.sum()
    return float((v[a].mean() - v[b].mean()) / math.sqrt(var)) if var > 0 else 0.0


def combine(r, zs):
    """신호별 z를 하나로: (z_Value + z_bad) ÷ √(2 + 2r). good_bad가 없으면 z_Value 그대로"""
    return zs[0] if len(zs) == 1 else (zs[0] + zs[1]) / math.sqrt(2 + 2 * r['r'])


def mean_z(d, r, m):
    """웨이퍼 묶음이 전체보다 나쁜지의 결합 z (조합의 z와 같은 계산)"""
    if m.sum() < 2:
        return 0.0
    zs = [sig_mean_z(d, r, k_, m) for k_ in r['signals']]
    if len(zs) > 1:
        bad01 = np.nan_to_num(d['bad'])
        zs[1] *= float(binom_fix([m.sum()], [bad01[m].sum()], bad01.mean())[0])
    return combine(r, zs)


def diff_z(d, r, a, b):
    """두 웨이퍼 묶음의 차이(a가 b보다 나쁜지)의 결합 z"""
    return combine(r, [sig_diff_z(d, r, k_, a, b) for k_ in r['signals']]) if a.sum() >= 2 and b.sum() >= 2 else 0.0


def classify(res, zk, d):
    """판정: 정상 · 경고 · 불량 · 상속 · 하위 기인 (웹 js/analysis.js classify와 같은 규칙).
    부모와의 비교(리프트)와 하위 기인은 기준을 넘은 조합에만 필요해서, 그 조합들만 웨이퍼로 직접 계산한다"""
    combos, by_key = res['combos'], res['by_key']
    mem = {}

    def member(c):
        if c['key'] not in mem:
            mem[c['key']] = members_of(d, c)
        return mem[c['key']]

    for c in combos:
        c['over'] = c['z'] > zk[c['k']]
        c['explained_by'], c['children'] = None, []
        c['status'] = ('bad' if c['k'] == 1 else None) if c['over'] else ('warn' if c['z'] > Z95 else 'normal')
    for c in combos:
        if not c['over'] or c['k'] == 1:
            continue
        m = member(c)
        c['parents'] = []
        for i in range(c['k']):
            p_items = c['items'][:i] + c['items'][i + 1:]
            pm = members_of(d, {'step': c['step'], 'items': p_items})
            rest = pm & ~m
            c['parents'].append({'key': combo_key(c['step'], p_items), 'items': p_items, 'n': int(pm.sum()), 'mean': float(d['Y'][pm].mean()),
                                 'rest_n': int(rest.sum()), 'rest_mean': float(d['Y'][rest].mean()) if rest.any() else math.nan,
                                 't': diff_z(d, res, m, rest)})
            if (P := by_key.get(c['parents'][-1]['key'])) is not None:
                P['children'].append(c['id'])
        c['lift_t'] = min(p['t'] for p in c['parents'])
        c['status'] = 'bad' if c['lift_t'] > zk[c['k']] else 'inherited'
    for k in range(res['max_depth'] - 1, 0, -1):
        for c in combos:
            if c['k'] != k or c['status'] != 'bad':
                continue
            for cid in c['children']:
                ch = combos[cid]
                if ch['status'] not in ('bad', 'explained'):
                    continue
                rest = member(c) & ~member(ch)
                if rest.sum() >= 2 and mean_z(d, res, rest) <= zk[k]:
                    c['status'], c['explained_by'] = 'explained', ch['id']
                    break
    count = lambda st: sum(c['status'] == st for c in combos)
    return {'over': sum(c['over'] for c in combos), 'bad': count('bad'), 'inherited': count('inherited'), 'explained': count('explained')}


def build_index(r):
    """기준 밖 조합을 (스텝, 오더, 유닛)으로 바로 찾는 색인. 후보 찾기가 전체 조합을 매번 훑지 않게 한다"""
    idx = defaultdict(list)
    for b in r['combos']:
        if b['over']:
            for it in b['items']:
                idx[(b['step'], it)].append(b)
    r['over_index'] = idx

In [ ]:
RANK_NAME = '초과 bad 웨이퍼' if data['B'] is not None else '영향 (웨이퍼 × 차이)'   # 랭킹 · 대표 선택 기준
STEP_NAME = None if STEP is None else re.sub(STEP_SUFFIX, '', str(STEP)) if STEP_SUFFIX else str(STEP)   # 'ex100000_3'을 줘도 스텝 ex100000
in_scope = lambda c: STEP_NAME is None or steps[c['step']]['name'] == STEP_NAME


def path_label(st, items, sep=' → '):
    return sep.join(f"{st['seqs'][q]['name']}:{st['seqs'][q]['units'][u]}" for q, u in items)


def metric(c, r):
    """랭킹 · 대표 선택 기준: 초과 bad 웨이퍼 = 웨이퍼 × (bad 비율 − 전체 bad 비율), 즉 고치면 줄어드는 bad 웨이퍼 수.
    good_bad가 없으면 웨이퍼 × y_value 차이"""
    return c['n'] * (c['bad_rate'] - r['p0']) if r['p0'] is not None else c['n'] * (c['mean'] - r['mu'])


def candidates(r, c):
    """고른 조합과 같은 스텝에서 유닛을 하나 이상 공유하고 기준을 넘은 조합 (색인으로 찾음)"""
    found = {b['id']: b for it in c['items'] for b in r['over_index'].get((c['step'], it), ())}
    return [found[i] for i in sorted(found)]


def representative(r, zk_, c):
    """기준 밖 점의 대표 불량 대상: 같은 스텝에서 유닛을 공유하는 불량 조합 중 초과 bad 웨이퍼가 가장 많은 것"""
    bad = [b for b in candidates(r, c) if b['status'] == 'bad']
    return max(bad, key=lambda b: metric(b, r)) if bad else None


def reason(r, st, b, target):
    """후보가 혐의 대상과 어떤 관계인지 쉬운 말로"""
    if b is target:
        return '혐의 대상'
    if b['status'] == 'bad':
        return '따로 불량 (다른 혐의 대상)'
    if b['status'] == 'explained':
        return f"더 좁은 조합이 원인 ({path_label(st, r['combos'][b['explained_by']]['items'])})"
    pr = min(b['parents'], key=lambda p: p['t'])
    return f"혐의 대상에 포함 ({path_label(st, pr['items'])}만으로 설명됨)"


def rank_mark(i):
    """funnel에 다는 순위 표시: ①②③ …"""
    return chr(0x2460 + i - 1) if 1 <= i <= 20 else str(i)


def fmt_y(v):
    """Value 표시: 전체 평균의 크기에 맞춘 자릿수로"""
    ref = abs(float(np.nanmean(data['value'])))
    return f"{v:,.{0 if ref >= 100 else 1 if ref >= 10 else 2 if ref >= 1 else 3}f}"


def target_row(c):
    """혐의 대상 표의 한 줄 (원래 단위)"""
    st, m = steps[c['step']], members_of(data, c)
    row = {'STEP': st['name'], '경로': path_label(st, c['items']), '웨이퍼': f"{c['n']:,}"}
    if data['B'] is not None:
        row[f"bad 비율 (전체 {res['p0']:.1%})"] = f"{np.nanmean(data['bad'][m]):.1%}"
        row['초과 bad (장)'] = f"{metric(c, res):+.0f}"
    row[f"Value 평균 (전체 {fmt_y(np.nanmean(data['value']))})"] = fmt_y(np.nanmean(data['value'][m]))
    row['확실도'] = f"{c['z'] / zk[c['k']]:.2f}"
    return row


def cand_row(b, target):
    """자세히 보기의 후보 표 한 줄"""
    st, m = steps[b['step']], members_of(data, b)
    row = {'경로': path_label(st, b['items']), '웨이퍼': b['n']}
    if data['B'] is not None:
        row['bad 비율'] = f"{np.nanmean(data['bad'][m]):.1%}"
    return row | {'확실도': f"{b['z'] / zk[b['k']]:.2f}", '관계': reason(res, st, b, target)}


def combo_row(d, r, zk_, c):
    st = d['steps'][c['step']]
    m = members_of(d, c)
    desc = ' / '.join(dict.fromkeys(st['seqs'][q]['desc'] for q, _ in c['items']))   # 경로가 지난 오더(step_seq)의 step_desc
    row = {'STEP': st['name'], 'step_desc': desc, '경로': path_label(st, c['items']), 'Order 수': c['k'], '웨이퍼': c['n']}
    row |= {'랏 수': len(np.unique(d['lots'][m])), f'평균 {Y_NAME}': c['mean']}
    if Y_NAME != 'Value':                                    # 값을 바꿔 분석했으면 원래 Value 평균도 함께
        row['평균 Value (원래 값)'] = float(np.nanmean(d['value'][m]))
    if d['B'] is not None:
        row['bad 비율'] = float(np.nanmean(d['bad'][m]))
    row[RANK_NAME] = round(metric(c, r), 1)
    row |= {'차이': c['mean'] - r['mu'], 'z ÷ 기준 z': c['z'] / zk_[c['k']], 'z Value': c['z_y']}
    if d['B'] is not None:
        row['z bad'] = c['z_b']
    return row


def cross_step_check(d, r, zk_):
    """스텝 간 규칙: 다른 스텝의 불량 조합 A와 웨이퍼가 겹치는 불량 조합 B에 대해, A를 지나지 않은 웨이퍼만 놓고
    B가 나머지보다 높은지(Welch t, 조합의 z와 같은 보정) 본다. B의 효과가 사라지고(t ≤ 1.96 × 퍼짐 배수) A는 B 없이도 기준을 넘으면 B를 'A 기인'으로 본다"""
    bads = [c for c in r['combos'] if c['status'] == 'bad']
    mem = {c['key']: members_of(d, c) for c in bads}

    def t_without(c, other):
        return diff_z(d, r, mem[c['key']] & ~mem[other['key']], ~mem[c['key']] & ~mem[other['key']])

    out = {}
    if not bads:
        return out
    M = np.array([mem[c['key']] for c in bads], dtype=np.float32)
    overlap = (M @ M.T) > 0                       # 웨이퍼가 겹치는 쌍을 한 번에
    for i, b in enumerate(bads):
        cause = [(t_without(b, a), a) for j, a in enumerate(bads) if overlap[i, j] and a['step'] != b['step']]
        cause = [(t, a) for t, a in cause if t <= Z95 * r['spread'] and t_without(a, b) > zk_[a['k']]]
        if cause:
            out[b['key']] = min(cause, key=lambda ta: ta[0])[1]
    return out

In [ ]:
CHART_SEP = ' ⟶ '       # 차트 안 경로 구분자. 나눔고딕의 '→'는 아주 작게 찍혀서 차트에서는 긴 화살표를 쓴다 (표는 '→')


def mix(c1, c2, t):
    a, b = np.array(to_rgb(c1)), np.array(to_rgb(c2))
    return tuple(a * t + b * (1 - t))


def fmt_p(p):
    if p is None or not math.isfinite(p):
        return '–'
    return f'{p:.0e}' if p < 1e-4 else f'{p:.4f}'


def plain_log(axis, wide=False):
    """로그 축 눈금을 보통 숫자로. 범위가 넓으면(wide) 10배마다 하나씩만"""
    axis.set_major_locator(LogLocator(base=10, subs=(1,) if wide else (1, 2, 5)))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}' if v >= 1 else f'{v:g}'))
    axis.set_minor_locator(NullLocator())


def label_points(ax, pts, gap=14):
    """불량 점 오른쪽에 이름. 점이 몰려 있으면 라벨을 무리의 오른쪽 바깥에 쌓고 가는 선으로 잇는다"""
    if not pts:
        return
    ax.autoscale_view()
    disp = ax.transData.transform([(x, y) for x, y, _ in pts])
    placed = []
    for i in np.argsort(disp[:, 1]):
        x0, y0 = disp[i]
        near = [j for j in range(len(pts)) if abs(disp[j][0] - x0) < 70 and abs(disp[j][1] - y0) < 20]
        lx, ly = max(disp[j][0] for j in near) + 9, y0
        for px, py in sorted(placed, key=lambda p: p[1]):
            if abs(lx - px) < 70 and abs(ly - py) < gap:
                ly = py + gap
        placed.append((lx, ly))
        x, y, text = pts[i]
        moved = len(near) > 1 or ly != y0
        ax.annotate(text, (x, y), xytext=(lx - x0 + (6 if moved else 0), ly - y0), textcoords='offset pixels', va='center',
                    fontsize=9, fontweight='bold', color=C['ink'], zorder=6,
                    arrowprops={'arrowstyle': '-', 'color': C['ink2'], 'lw': 0.8, 'shrinkA': 1, 'shrinkB': 5} if moved else None)


def plot_unified(ax, d, r, zk_, cross=None, labels=None, title=''):
    """모든 STEP · 모든 조합 크기를 한 장의 funnel로. 가로축은 조합의 웨이퍼 수(웨이퍼 한 장 = 1),
    띠는 기준선 μ ± z₁·σ/√n·√(1 − n/전체 웨이퍼 수)이다. 점의 높이는 결합 z ÷ z_k만큼 띠에 맞춰 그려서
    띠 하나로 모든 크기를 판정한다(띠 밖 ⇔ z > z_k). labels의 조합(혐의 대상)에만 순위를 단다"""
    cross = cross or {}
    cs = [c for c in r['combos'] if in_scope(c)]
    mu_, z1 = r['mu'], zk_[1]
    half = lambda n: z1 * r['sd'] / np.sqrt(n) * shrink(n, r['N'])

    def xy(group):
        if not group:
            return np.array([]), np.array([])
        n = np.array([c['n'] for c in group], dtype=float)
        return n, mu_ + np.array([c['z'] / zk_[c['k']] for c in group]) * half(n)

    pt = lambda c: (c['n'], float(xy([c])[1][0]))
    ns = [c['n'] for c in cs]
    lo, hi = max(2, min(ns) * 0.9), min(max(ns) * 1.1, r['N'])
    grid = lo * (hi / lo) ** (np.arange(80) / 79)
    ax.fill_between(grid, mu_ - half(grid), mu_ + half(grid), color=C['band'], lw=0, zorder=0)
    for sign in (1, -1):
        ax.plot(grid, mu_ + sign * half(grid), color=C['axis'], lw=1, zorder=1)
    ax.plot([lo, hi], [mu_, mu_], color=C['axis'], lw=1, ls=(0, (4, 3)), zorder=1)
    ax.scatter(*xy([c for c in cs if not c['over']]), s=4, color=C['muted'], alpha=0.22, lw=0, zorder=1)
    over = [c for c in cs if c['over']]
    ax.scatter(*xy([c for c in over if c['status'] != 'bad']), s=46, facecolors='none', edgecolors=C['bad'], lw=1.4, zorder=3)
    ax.scatter(*xy([c for c in over if c['status'] == 'bad' and c['key'] not in cross]), s=80, color=C['bad'], edgecolors=C['surface'], lw=1.8, zorder=4)
    xc = [c for c in cs if c['key'] in cross]
    ax.scatter(*xy(xc), s=110, facecolors=C['surface'], edgecolors=C['bad'], lw=2.4, zorder=5)
    if LOGX:
        ax.set_xscale('log')
        plain_log(ax.xaxis)
    steps_ = d['steps']
    label_points(ax, [(*pt(r['by_key'][k]), text) for k, text in (labels or {}).items() if in_scope(r['by_key'][k])])
    handles = [Line2D([], [], marker='o', ls='', color=C['bad'], mec=C['surface'], ms=8, label='불량 판정'),
               Line2D([], [], marker='o', ls='', mfc='none', mec=C['bad'], ms=7, label='같이 올라온 조합'),
               Line2D([], [], marker='o', ls='', color=C['muted'], ms=4, label='기준선 안 조합'),
               Patch(color=C['band'], label='기준선 범위')]
    if xc:
        handles.append(Line2D([], [], marker='o', ls='', mfc=C['surface'], mec=C['bad'], mew=2.2, ms=9, label='다른 STEP 탓 의심'))
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0)
    ax.set(title=title, xlabel='조합의 웨이퍼 수', ylabel='Value')
    ax.tick_params(axis='y', labelleft=False)          # 세로 위치는 기준선에 맞춘 환산값이라 눈금 값은 적지 않는다
    if not HIGHER_IS_WORSE:
        ax.invert_yaxis()                               # 수율처럼 클수록 좋은 값은 낮은 조합이 아래에 오게
    ax.grid(axis='x', visible=False)


def plot_flow(c):
    """STEP SEQ: Order × 유닛 박스를 유닛의 bad 비율(good_bad가 없으면 평균)로 색칠하고 대상 경로를 잇는다"""
    st, A = steps[c['step']], data['assign'][c['step']]
    hit = dict(c['items'])
    has_b = data['B'] is not None
    base = float(np.nanmean(data['bad'])) if has_b else res['mu']

    def level(q):
        a = A[q]
        ok, nu = a != MISSING, len(st['seqs'][q]['units'])
        v, n_ = (data['bad'] if has_b else data['Y']), np.bincount(a[ok], minlength=nu)
        return np.divide(np.bincount(a[ok], weights=v[ok], minlength=nu), n_, out=np.full(nu, np.nan), where=n_ > 0)

    qs_all = list(range(len(st['seqs'])))
    lv = {q: level(q) for q in qs_all}
    shown = qs_all
    if len(qs_all) > FLOW_MAX_ORDERS:                   # 오더가 많으면 경로 → 유닛 간 차이가 큰 오더 순으로 고른다
        gap_of = lambda q: float(np.nanmax(lv[q]) - np.nanmin(lv[q])) if np.isfinite(lv[q]).sum() >= 2 else -1.0
        rest = sorted((q for q in qs_all if q not in hit), key=lambda q: -gap_of(q))
        shown = sorted(list(hit) + rest[:max(0, FLOW_MAX_ORDERS - len(hit))])
    text_of = lambda q, ui: st['seqs'][q]['units'][ui] + (f"  {lv[q][ui]:.0%}" if has_b and np.isfinite(lv[q][ui]) else '')
    pos = {q: i for i, q in enumerate(shown)}
    maxlen = max(len(text_of(q, ui)) for q in shown for ui in range(len(st['seqs'][q]['units'])))
    colW, gap, boxH, boxGap, top, padX = max(84, 7 * maxlen + 18), 26, 22, 4, 26, 2
    nq, maxU = len(shown), max(len(st['seqs'][q]['units']) for q in shown)
    W = padX * 2 + nq * colW + (nq - 1) * gap
    H = top + maxU * (boxH + boxGap) + 4
    max_dev = max([1e-9] + [float(np.nanmax(lv[q] - base)) for q in shown if np.isfinite(lv[q]).any()])
    col_x = lambda q: padX + pos[q] * (colW + gap)
    box_y = lambda u: top + u * (boxH + boxGap)
    fig = plt.figure(figsize=(W / 72, (H + 26) / 72))
    ax = fig.add_axes((0, 0, 1, H / (H + 26)))
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.axis('off')
    note = f' · Order {len(qs_all)}개 중 {len(shown)}개' if len(shown) < len(qs_all) else ''
    fig.text(0, 1, f"STEP SEQ · {st['name']} · 색이 진할수록 {'bad 비율' if has_b else 'Value'}이 높음{note}",
             ha='left', va='top', fontsize=11, fontweight='bold', color=C['ink'])
    for q in shown:
        on, cx = q in hit, col_x(q) + colW / 2
        ax.text(cx, 15, f"Order {st['seqs'][q]['name'][1:]}", ha='center', va='baseline', fontsize=12, fontweight='bold', color=C['bad'] if on else C['ink2'])
        for ui in range(len(st['seqs'][q]['units'])):
            sel = on and hit[q] == ui
            v = lv[q][ui]
            t = 0.0 if not np.isfinite(v) else min(1.0, max(0.0, (v - base) / max_dev))
            fill = C['bad'] if sel else mix(C['bad'], C['mid'], round(t * 55) / 100)
            alpha = 1 if on else 0.45
            ax.add_patch(FancyBboxPatch((col_x(q), box_y(ui)), colW, boxH, boxstyle='round,pad=0,rounding_size=6',
                                        fc=fill, ec='none', alpha=alpha, zorder=4 if sel else 3))
            ax.text(cx, box_y(ui) + boxH / 2 + 4, text_of(q, ui), ha='center', va='baseline', fontsize=10,
                    fontweight='bold' if sel else 'normal', color='white' if sel else C['ink'], alpha=alpha, zorder=5)
    pts = sorted((col_x(q) + colW / 2, box_y(u) + boxH / 2) for q, u in c['items'])
    if len(pts) > 1:
        verts, codes = [pts[0]], [MplPath.MOVETO]
        for (x0, y0), (x1, y1) in zip(pts, pts[1:]):
            dx = (x1 - x0) / 2
            verts += [(x0 + dx, y0), (x1 - dx, y1), (x1, y1)]
            codes += [MplPath.CURVE4] * 3
        ax.add_patch(PathPatch(MplPath(verts, codes), fc='none', ec=C['bad'], lw=3, alpha=0.85, capstyle='round', zorder=2))
    plt.show()


def n_values(T):
    """good_bad가 N인 웨이퍼 중 이 경로 · 나머지를 지난 웨이퍼의 Value (차트 표시용, 계산에는 쓰지 않음)"""
    st = steps[T['step']]
    want = {st['seqs'][q]['ord']: st['seqs'][q]['units'][u] for q, u in T['items']}
    g = h_n[(h_n['step'] == st['name']) & h_n['ord'].isin(list(want))]
    if not len(g):
        return np.array([]), np.array([])
    tab = g.pivot(index='wafer_key', columns='ord', values='unit').reindex(columns=list(want)).dropna()
    on = np.logical_and.reduce([tab[o].astype(str).to_numpy() == u for o, u in want.items()])
    v = n_value.reindex(tab.index).to_numpy(float)
    return v[on], v[~on]


def plot_compare(T):
    """이 경로 vs 나머지(같은 Order들을 지났지만 이 경로가 아닌 웨이퍼): bad 비율과 원래 Value 분포.
    good_bad가 N인 웨이퍼는 계산에서 빼고 Value 분포에 빈 원으로만 그린다"""
    A = data['assign'][T['step']][[q for q, _ in T['items']]]
    me = members_of(data, T)
    groups, names, cols = [me, np.all(A != MISSING, axis=0) & ~me], ['이 경로', '나머지'], [C['bad'], C['muted']]
    nv = n_values(T)
    has_b = data['B'] is not None
    fig, axes = plt.subplots(1, 2 if has_b else 1, figsize=(13, 2.4), gridspec_kw={'width_ratios': [1, 1.7]} if has_b else None)
    axes = np.atleast_1d(axes)
    if has_b:
        ax = axes[0]
        rates = [float(np.nanmean(data['bad'][g])) if g.any() else 0.0 for g in groups]
        ax.barh([0, 1], rates, color=cols, height=0.55)
        for yy, (rt, g, v_n) in enumerate(zip(rates, groups, nv)):
            ax.text(rt, yy, f'  {rt:.1%} ({int(g.sum()):,}장' + (f' · N {len(v_n):,}장 제외)' if len(v_n) else ')'),
                    va='center', fontsize=10, color=C['ink'])
        ax.set_xlim(0, max(rates + [0.01]) * 1.6)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.0%}'))
        ax.set_title('bad 비율')
    ax = axes[-1]
    vals = [data['value'][g][np.isfinite(data['value'][g])] for g in groups]
    ok = [v for v in vals + list(nv) if len(v)]
    logx = bool(ok) and all(v.min() > 0 for v in ok) and pd.Series(np.concatenate(ok)).skew() > 1
    horizontal = {'orientation': 'horizontal'} if tuple(map(int, mpl.__version__.split('.')[:2])) >= (3, 10) else {'vert': False}
    bp = ax.boxplot([v if len(v) else [np.nan] for v in vals], **horizontal, positions=[0, 1], widths=0.5, showfliers=False,
                    patch_artist=True, medianprops={'color': C['ink'], 'lw': 2, 'zorder': 4}, whiskerprops={'color': C['axis']}, capprops={'color': C['axis']})
    for patch, col in zip(bp['boxes'], cols):
        patch.set(facecolor=mix(col, C['surface'], 0.18), edgecolor=col, zorder=2)
    rng_ = np.random.default_rng(0)
    for yy, (v, col) in enumerate(zip(vals, cols)):              # 웨이퍼 점 (많으면 800장만)
        v = v if len(v) <= 800 else rng_.choice(v, 800, replace=False)
        ax.scatter(v, yy + (rng_.random(len(v)) - 0.5) * 0.36, s=7, color=col, alpha=0.4, lw=0, zorder=3)
    for yy, (v, col) in enumerate(zip(nv, cols)):                # good_bad N 웨이퍼: 빈 원 (계산에는 안 씀)
        v = v if len(v) <= 300 else rng_.choice(v, 300, replace=False)
        ax.scatter(v, yy + (rng_.random(len(v)) - 0.5) * 0.36, s=16, facecolors='none', edgecolors=col, lw=0.9, alpha=0.8, zorder=3)
    for yy, v in enumerate(vals):
        if len(v):
            ax.annotate(f'중앙값 {np.median(v):.3g}', (np.median(v), yy - 0.25), xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=9, color=C['ink'])
    if logx:
        ax.set_xscale('log')
        allv = np.concatenate(ok)
        plain_log(ax.xaxis, wide=allv.max() / allv.min() > 1e3)
    ax.set_title('Value 분포 · 점 = 웨이퍼 · 상자 = 가운데 50%' + (' · 빈 원 = good_bad N(계산 제외)' if any(len(v) for v in nv) else '')
                 + (' · 로그 눈금' if logx else ''))
    for ax in axes:
        ax.set_yticks([0, 1], names)
        ax.set_ylim(1.6, -0.6)
        ax.grid(axis='y', visible=False)
    plt.tight_layout()
    plt.show()


def plot_time(T):
    """시간 추이: 이 STEP의 track-in 시각 순서로 이 경로와 STEP 전체의 이동 bad 비율 (good_bad가 없으면 Value 이동 중앙값)"""
    s = T['step']
    t = data['trackin'][s]
    present = np.any(data['assign'][s] != MISSING, axis=0) & ~np.isnat(t)
    me = members_of(data, T) & ~np.isnat(t)
    if me.sum() < 8:
        print('track-in 시각이 있는 웨이퍼가 적어 시간 추이를 건너뜁니다.')
        return
    has_b = data['B'] is not None
    v = data['bad'] if has_b else data['value']
    all_s = pd.Series(v[present], index=pd.DatetimeIndex(t[present])).sort_index()
    me_s = pd.Series(v[me], index=pd.DatetimeIndex(t[me])).sort_index()
    win_all, win_me = max(20, min(250, len(all_s) // 20)), max(8, min(60, round(len(me_s) / 8)))
    roll = (lambda x, w: x.rolling(w).mean()) if has_b else (lambda x, w: x.rolling(w).median())
    fig, ax = plt.subplots(figsize=(13, 3.2))
    ax.plot(roll(all_s, win_all), color=C['muted'], lw=2, label=f'STEP 전체 (웨이퍼 {win_all}장씩)')
    ax.plot(roll(me_s, win_me), color=C['bad'], lw=2.5, label=f'이 경로 (웨이퍼 {win_me}장씩)')
    if has_b:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.set(title=f"시간 추이 · STEP {steps[s]['name']} · track-in 시각 · {'bad 비율' if has_b else 'Value 중앙값'}")
    ax.legend(loc='lower right', bbox_to_anchor=(1, 1.0), ncol=2, borderaxespad=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def envelope_points():
    """경계선 비교용: 범위 안의 조합마다 웨이퍼 수 N, 조합 평균, Value z, Order 수, 같은 N의 점 수 M_N"""
    cs = [c for c in res['combos'] if in_scope(c)]
    n = np.array([c['n'] for c in cs], dtype=float)
    _, inv, cnt = np.unique(n, return_inverse=True, return_counts=True)
    return {'n': n, 'y': np.array([c['mean'] for c in cs]), 'z': np.array([c['z_y'] for c in cs]),
            'k': np.array([c['k'] for c in cs]), 'M': cnt[inv]}


def plot_envelope(kind):
    """같은 조합에 한 가지 경계선을 그린다. 두 경계 모두 σ_N = σ/√N·√(1 − N/전체 웨이퍼 수)에 배수를 곱하고, 배수를 정하는 M만 다르다.
    σ_N은 레퍼런스 코드의 σ̂_n(웨이퍼 N장씩 무작위로 뽑은 평균들의 표준편차, 비복원)과 같은 값을 식으로 구한 것이다(뽑기 잡음 없음).
    'bonf': 배수 = 퍼짐 배수 × Φ⁻¹(1 − 0.05/D/M), D = 조합이 있는 Order 수, M = 같은 Order 수의 전체 조합 수 (지금 판정) · 가장 낮은 · 높은 기준선과 그 사이 범위만 그린다
    'bonf1': 'bonf'와 같은 판정을 3절처럼 그린다. 점 높이를 자기 기준 대비 비율로 바꿔 Order 1개 기준 띠 하나로 보인다
    'memo': 배수 = √(2 ln M_N), M_N = 같은 N(웨이퍼 수)에 찍힌 점 수 (레퍼런스 식 그대로, 퍼짐 보정 없음) · N마다 계단 모양.
            레퍼런스처럼 M_N < 2인 N은 경계를 그리지 않고 판정하지 않으며, 아래 칸에 N마다 M_N을 그린다"""
    e = envelope_points()
    sign = 1 if HIGHER_IS_WORSE else -1                 # 클수록 좋은 Value는 원래 부호로 그린다 (나쁜 쪽이 아래)
    mu, y = sign * res['mu'], sign * e['y']
    grid, m_grid = np.unique(e['n'], return_counts=True)
    sd_n = res['sd'] / np.sqrt(grid) * shrink(grid, res['N'])   # 웨이퍼 N장 평균이 전체 평균에서 우연히 벗어나는 폭
    if kind == 'bonf':
        fig, ax = plt.subplots(figsize=(13, 5))
        out = np.abs(e['z']) > np.array([zk[k] for k in e['k']])     # 점마다 자기 Order 수의 기준으로 판정
        ks = [k for k in range(1, MAX_DEPTH + 1) if res['search_space'][k]]
        k_lo, k_hi = min(ks, key=lambda k: zk[k]), max(ks, key=lambda k: zk[k])   # 가장 낮은 · 높은 기준선만 그린다
        for sgn in (1, -1):
            ax.fill_between(grid, mu + sgn * zk[k_lo] * sd_n, mu + sgn * zk[k_hi] * sd_n, color=C['band'], lw=0, zorder=0,
                            label='칠한 범위: Order 수에 따라 안 · 밖이 갈림' if sgn == 1 and k_hi != k_lo else None)
            ax.plot(grid, mu + sgn * zk[k_lo] * sd_n, color=C['ink'], lw=1.6,
                    label=f"Order {k_lo}개 기준{' (가장 낮음)' if k_hi != k_lo else ''}: μ ± {zk[k_lo]:.2f}·σ_N" if sgn == 1 else None)
            if k_hi != k_lo:
                ax.plot(grid, mu + sgn * zk[k_hi] * sd_n, color=C['ink'], lw=1.6, ls=(0, (5, 3)),
                        label=f'Order {k_hi}개 기준 (가장 높음): μ ± {zk[k_hi]:.2f}·σ_N' if sgn == 1 else None)
        title = 'Bonferroni 경계 · 점마다 자기 Order 수의 기준으로 판정'
    elif kind == 'bonf1':
        fig, ax = plt.subplots(figsize=(13, 5))
        zkk = np.array([zk[k] for k in e['k']])
        out = np.abs(e['z']) > zkk                      # 판정은 'bonf'와 같다
        sd_pt = res['sd'] / np.sqrt(e['n']) * shrink(e['n'], res['N'])
        y = mu + sign * e['z'] / zkk * zk[1] * sd_pt   # 3절처럼: 점 높이 = 자기 기준 대비 비율 × Order 1개 기준 띠
        ax.fill_between(grid, mu - zk[1] * sd_n, mu + zk[1] * sd_n, color=C['band'], lw=0, zorder=0)
        for sgn in (1, -1):
            ax.plot(grid, mu + sgn * zk[1] * sd_n, color=C['ink'], lw=1.6,
                    label=f'기준선 띠: μ ± {zk[1]:.2f}·σ_N (Order 1개 기준)' if sgn == 1 else None)
        ax.scatter([], [], s=0, label='다른 Order 수는 점 높이를 자기 기준에 맞춤')
        title = 'Bonferroni 경계 · 3절처럼 점 높이를 자기 기준 대비 비율로 그림'
    else:
        fig, (ax, axm) = plt.subplots(2, 1, figsize=(13, 6.4), sharex=True, gridspec_kw={'height_ratios': [4, 1]})
        judged = e['M'] >= 2                            # 점이 하나뿐인 N은 퍼짐을 알 수 없어 경계 없음 (레퍼런스와 같음)
        out = judged & (np.abs(e['z']) > np.sqrt(2 * np.log(e['M'])))
        half = np.where(m_grid >= 2, np.sqrt(2 * np.log(m_grid)) * sd_n, np.nan)
        ax.plot(grid, mu + half, color=C['accent'], lw=1.4, drawstyle='steps-mid', label='μ ± √(2 ln M_N)·σ_N')
        ax.plot(grid, mu - half, color=C['accent'], lw=1.4, drawstyle='steps-mid')
        if (~judged).any():
            ax.scatter([], [], s=0, label=f'M_N = 1이라 경계 없음 {int((~judged).sum()):,}개')
        axm.vlines(grid, 0.8, m_grid, color=C['ink2'], lw=1)
        axm.set_yscale('log')
        plain_log(axm.yaxis, wide=True)
        axm.set(ylabel='M_N', xlabel='조합의 웨이퍼 수 N')
        axm.grid(axis='x', visible=False)
        title = '√(2 ln M_N) 경계 · M_N = 같은 N(웨이퍼 수)에 찍힌 점 수 (아래 칸)'
    up, lo = out & (e['z'] > 0), out & (e['z'] < 0)
    ax.scatter(e['n'][~out], y[~out], s=3, color=C['muted'], alpha=0.2, lw=0, zorder=1)
    ax.scatter(e['n'][up], y[up], s=18, color=C['bad'], lw=0, zorder=3, label=f'경계 밖 · 나쁜 쪽 {up.sum():,}개')
    ax.scatter(e['n'][lo], y[lo], s=18, color=C['ink2'], lw=0, zorder=3, label=f'경계 밖 · 좋은 쪽 {lo.sum():,}개')
    widest = max(float(np.nanmax(zk)), float(np.sqrt(2 * np.log(m_grid.max())))) * float(sd_n.max())   # 두 차트의 세로 범위를 같게
    span = max(float(np.abs(y - mu).max()), widest) * 1.05
    ax.set_ylim(mu - span, mu + span)
    if LOGX:
        ax.set_xscale('log')
        for a in fig.axes:
            plain_log(a.xaxis)
    ax.set(title=f'{title} · 조합 {len(e["n"]):,}개', ylabel=f"조합 평균 {Y_NAME.replace('−Value', 'Value')}")
    if kind == 'bonf1':
        ax.set_ylabel('Value (자기 기준에 맞춘 높이)')
        ax.tick_params(axis='y', labelleft=False)         # 환산한 높이라 눈금 값은 적지 않는다 (3절과 같음)
    if kind != 'memo':
        ax.set_xlabel('조합의 웨이퍼 수 N')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0)
    ax.grid(axis='x', visible=False)
    plt.tight_layout()
    plt.show()

## 2. 판정
판정 전에 Value 치우침과 part 차이를 자동으로 맞춥니다. 그다음 웨이퍼(`root_lot_id` + `wafer_id`) 한 장을 하나의 표본으로 보고, 모든 조합(STEP 안에서 Order 1~`MAX_DEPTH`개의 유닛을 지난 웨이퍼 묶음)을 같은 기준선으로 판정합니다. 계산값은 부록 B에 있습니다.

In [ ]:
# ① 분포 모양: y_value가 한쪽으로 길게 치우치면 순위 점수(순위를 정규분포 점수로 옮긴 값)로 바꾼다. 순서는 그대로
adjust_notes = []                                  # '보정:' 한 줄에 쓸 내용
skew = float(pd.Series(data['Y']).skew())
tf = ('rank' if abs(skew) > 1 else None) if Y_TRANSFORM == 'auto' else Y_TRANSFORM
assert tf in (None, 'rank', 'log'), f"Y_TRANSFORM은 'auto', 'rank', 'log', None 중 하나: {Y_TRANSFORM}"
if tf == 'rank':
    data['Y'] = -norm_isf((pd.Series(data['Y']).rank().to_numpy() - 0.5) / N)
    Y_NAME += ' 순위 점수'
    adjust_notes.append(f'Value 치우침(왜도 {skew:.1f}) → 순위로 비교')
elif tf == 'log':
    lo = data['Y'].min()
    data['Y'] = np.log(data['Y'] if lo > 0 else data['Y'] - lo + max(1e-9, 0.01 * float(np.median(data['Y'] - lo))))
    Y_NAME = f'log {Y_NAME}'
    adjust_notes.append(f'Value 치우침(왜도 {skew:.1f}) → 로그로 비교')
Y = data['Y']

# ② part별 수준 차이: y_value와 bad 비율 각각, part 간 차이가 뚜렷하면 웨이퍼 값에서 part 평균을 빼고 전체 평균을 더한다
part_table, part_lines = None, []
if len(set(data['parts'])) > 1 and PART_ADJUST is not False:
    checks, table = [], {}
    for key_, label in (('Y', Y_NAME), ('B', 'bad 비율')):
        v = data[key_]
        if v is None:
            continue
        pg = pd.DataFrame({'part': data['parts'], 'v': v}).groupby('part')['v']
        n_p, m_p = pg.size(), pg.mean()
        eta2 = float((n_p * (m_p - v.mean()) ** 2).sum()) / float(((v - v.mean()) ** 2).sum())
        an_p = anova(n_p.to_numpy(), pg.sum().to_numpy(), pg.apply(lambda x: (x * x).sum()).to_numpy())
        p_part = an_p['p'] if an_p else 1.0
        table['웨이퍼'], table[f'평균 {label}'] = n_p, m_p
        checks.append((key_, label, eta2, p_part, PART_ADJUST is True or (p_part < 0.001 and eta2 >= 0.01), pg))
    part_table = pd.DataFrame(table).sort_values('웨이퍼', ascending=False).head(10).rename_axis('part_id')
    done = []
    for key_, label, eta2, p_part, adjust, pg in checks:
        part_lines.append(f'part 간 {label} 차이: 설명 비율 {eta2:.1%} · p {fmt_p(p_part)} → ' + ('맞춤' if adjust else '그대로'))
        if adjust:
            data[key_] = data[key_] - pg.transform('mean').to_numpy() + data[key_].mean()
            done.append('Value' if key_ == 'Y' else 'bad')
    if done:
        adjust_notes.append(f"part 차이 → 맞춤({', '.join(done)})")
        Y_NAME += ' (part 보정)' if 'Value' in done else ''
    Y = data['Y']

# ③ 결합 z: 신호(Value, bad)마다 z를 구해 하나로 합친다
t0 = time.perf_counter()
res = analyze(data, min_n=MIN_N, max_depth=MAX_DEPTH)
analyze_sec = time.perf_counter() - t0
zk = thresholds(res)
if res['spread'] > 1:
    adjust_notes.append(f"z가 이론보다 {res['spread']:.2f}배 퍼짐 → 기준선도 {res['spread']:.2f}배로")
summary = classify(res, zk, data)
build_index(res)
print('보정: ' + (' · '.join(adjust_notes) if adjust_notes else '필요 없음'))

## 3. 한 장 funnel과 혐의 대상
- 점 하나 = 조합 하나. 가로축은 조합의 웨이퍼 수, 세로축은 Value입니다. 기준선(회색 띠) 밖의 빨간 점이 불량 판정이고, 원 숫자는 아래 표의 순위입니다.
- 세로 위치는 Order 수까지 반영해 기준선에 맞춘 값이라 눈금 숫자는 적지 않습니다. 실제 Value 평균은 표에 있습니다.
- **초과 bad**: 이 대상을 고치면 줄어드는 bad 웨이퍼 수. 표는 이 순서입니다.
- **확실도**: 기준선을 몇 배 넘었는지. 1 이상이면 불량 판정입니다.
- **비고**: 웨이퍼가 거의 같아 한 줄로 묶은 대상(데이터로는 어느 쪽 탓인지 가를 수 없음), 다른 STEP 탓으로 보이는 대상

In [ ]:
over_pts = [c for c in res['combos'] if c['over'] and in_scope(c)]
by_target = defaultdict(list)                     # 기준선 밖 점 → 혐의 대상(같은 STEP에서 유닛을 공유하는 불량 조합 중 초과 bad가 가장 큰 것)
for c in over_pts:
    t = representative(res, zk, c)
    by_target[t['key'] if t else None].append(c)
cross = cross_step_check(data, res, zk)
same_as, kept = {}, []                            # 웨이퍼가 거의 같은 대상은 순위가 높은 쪽 한 줄로
for c in sorted((res['by_key'][k] for k in by_target if k is not None), key=lambda c: -metric(c, res)):
    m = members_of(data, c)
    hit = next((k2 for k2, m2 in kept if SAME_WAFERS and (m & m2).sum() >= SAME_WAFERS * (m | m2).sum()), None)
    if hit is None:
        kept.append((c['key'], m))
    else:
        same_as[c['key']] = hit
top = [res['by_key'][k] for k, _ in kept]         # 혐의 대상 (순위 순)


def same_text(key):
    """한 줄로 묶인 대상의 경로. 같은 스텝이면 경로만, 다른 스텝이면 스텝과 경로"""
    s0 = res['by_key'][key]['step']
    return ', '.join(path_label(steps[b['step']], b['items']) if b['step'] == s0 else f"{steps[b['step']]['name']} {path_label(steps[b['step']], b['items'])}"
                     for b in (res['by_key'][k] for k, v in same_as.items() if v == key))


n_shown = sum(in_scope(c) for c in res['combos'])
print(f"판정: 조합 {n_shown:,}개 검사 → 기준선을 넘은 조합 {len(over_pts):,}개 → 혐의 대상 {len(top)}개"
      + (f" (기준선은 z 퍼짐 {res['spread']:.2f}배만큼 넓힘)" if res['spread'] > 1 else ''))
fig, ax = plt.subplots(figsize=(13, 5.2))
plot_unified(ax, data, res, zk, cross=cross, labels={c['key']: rank_mark(i + 1) for i, c in enumerate(top[:10])},
             title=f'모든 조합 {n_shown:,}개 · 기준선 밖 {len(over_pts):,}개')
plt.tight_layout()
plt.show()

rep_df = pd.DataFrame()
if top:
    notes = []
    for c in top:
        note = [f"같은 웨이퍼: {same_text(c['key'])}"] if same_text(c['key']) else []
        if (a := cross.get(c['key'])) is not None:
            note.append(f"다른 STEP 탓 의심: {steps[a['step']]['name']} {path_label(steps[a['step']], a['items'])}")
        notes.append(' · '.join(note))
    view = pd.DataFrame([target_row(c) for c in top], index=pd.RangeIndex(1, len(top) + 1, name='순위'))
    if any(notes):
        view['비고'] = notes
    display(view)
    rep_df = pd.DataFrame([{**combo_row(data, res, zk, c), '같은 웨이퍼 대상': same_text(c['key']) or '–',
                            '기준선 밖 점': len(by_target[c['key']]) + sum(len(by_target[a]) for a, v in same_as.items() if v == c['key']),
                            '비고': note or '–', 'key': c['key']} for c, note in zip(top, notes)],
                          index=pd.RangeIndex(1, len(top) + 1, name='순위'))
    if SAVE_RANKING:
        rep_df.to_csv(SAVE_RANKING, encoding='utf-8-sig')
        print(f'혐의 대상 표(모든 열) 저장: {SAVE_RANKING}')
else:
    print('기준선을 넘은 혐의 대상이 없습니다.')

**Order 수별 판정 (검증용)**: 기준선을 넘은 조합이 Order 수마다 어떻게 정리됐는지 셉니다. Order 2개 이상에서 '불량 판정'이 0이면 여러 Order 조합은 모두 유닛 하나로 설명된 것('혐의 대상에 포함')이고, 0이 아닌데 '표의 혐의 대상'이 0이면 유닛을 공유하면서 초과 bad가 더 큰 대상에 묶인 것입니다.

In [ ]:
over_view = [c for c in res['combos'] if c['over'] and in_scope(c)]
check = (pd.crosstab(pd.Series([c['k'] for c in over_view], name='Order 수', dtype=int), pd.Series([c['status'] for c in over_view], dtype=str))
         .reindex(index=range(1, MAX_DEPTH + 1), columns=['bad', 'inherited', 'explained'], fill_value=0)
         .rename(columns={'bad': '불량 판정', 'inherited': '혐의 대상에 포함', 'explained': '더 좁은 조합이 원인'}))
check.insert(0, '기준선 밖', check.sum(axis=1))
check['표의 혐의 대상'] = pd.Series(Counter(c['k'] for c in top), dtype=int).reindex(check.index, fill_value=0)
display(check.rename_axis('Order 수').rename_axis(None, axis=1))

## 4. 혐의 대상 자세히
표의 `PICK`위 대상을 봅니다. 후보 표는 같은 STEP에서 기준선을 넘은 조합이 이 대상과 어떤 관계인지 보여 줍니다.
차트는 STEP SEQ(유닛마다 bad 비율) · 이 경로 vs 나머지(같은 Order들을 지났지만 이 경로가 아닌 웨이퍼) · 시간 추이 순서입니다.

In [ ]:
if top:
    T = top[min(PICK, len(top)) - 1]
    st, m = steps[T['step']], members_of(data, T)
    desc = ' / '.join(dict.fromkeys(st['seqs'][q]['desc'] for q, _ in T['items']))
    print(f"{min(PICK, len(top))}위 · STEP {st['name']} ({desc}) · {path_label(st, T['items'])}")
    print(f"웨이퍼 {T['n']:,}장"
          + (f" · bad {np.nanmean(data['bad'][m]):.1%} (전체 {res['p0']:.1%}, 초과 {metric(T, res):+.0f}장)" if data['B'] is not None else '')
          + f" · Value 평균 {fmt_y(np.nanmean(data['value'][m]))} (전체 {fmt_y(np.nanmean(data['value']))}) · 확실도 {T['z'] / zk[T['k']]:.2f}")
    print('Order: ' + ' · '.join(f"{st['seqs'][q]['name']} = {st['seqs'][q]['seq']}" for q, _ in T['items'])
          + (f" · 같은 웨이퍼: {same_text(T['key'])}" if same_text(T['key']) else ''))
    cands = sorted(candidates(res, T), key=lambda b: -metric(b, res))
    display(pd.DataFrame([cand_row(b, T) for b in cands], index=pd.RangeIndex(1, len(cands) + 1, name='후보')))

In [ ]:
if top:
    plot_flow(T)
    plot_compare(T)
    plot_time(T)

## 5. 다른 STEP 탓 의심
두 STEP의 설비가 짝지어 운영되면 한 원인이 여러 STEP에서 불량으로 보일 수 있습니다. 상대 STEP의 웨이퍼를 빼면 효과가 사라지는 혐의 대상을 적습니다.

In [ ]:
shown_cross = [(res['by_key'][c['key']], cross[c['key']]) for c in top if c['key'] in cross]
if shown_cross:
    display(pd.DataFrame([{'혐의 대상': f"{steps[b['step']]['name']} {path_label(steps[b['step']], b['items'])}",
                           '원인으로 보이는 조합': f"{steps[a['step']]['name']} {path_label(steps[a['step']], a['items'])}"}
                          for b, a in shown_cross], index=pd.RangeIndex(1, len(shown_cross) + 1, name='번호')))
else:
    print('다른 STEP 탓으로 보이는 혐의 대상은 없습니다.')

## 6. 경계선 비교: Bonferroni vs √(2 ln M_N)
같은 조합(점)에 두 경계선을 따로 그립니다. 두 경계 모두 σ_N(웨이퍼 N장 평균이 전체 평균에서 우연히 벗어나는 폭 = σ/√N × √(1 − N/전체 웨이퍼 수))에 배수를 곱하고, **배수를 정하는 M만** 다릅니다. σ_N은 웨이퍼 N장씩 무작위로 여러 번 뽑은 평균들의 표준편차와 같은 값이고, 뽑는 대신 식으로 구해 뽑기 잡음이 없습니다.
- **Bonferroni (지금 판정)**: 배수 = 퍼짐 배수(부록 C) × Φ⁻¹(1 − 0.05 ÷ Order 수 개수 ÷ M), M = 같은 Order 수의 전체 조합 수. 5%를 Order 수마다 나눠 써서 차트 전체에서 우연히 하나라도 넘을 확률이 5%입니다. Order 수마다 기준이 조금씩 달라서 가장 낮은 선과 가장 높은 선만 그리고 그 사이를 칠했습니다. 점은 자기 Order 수의 기준으로 판정하므로, 칠한 범위 안의 점은 Order 수에 따라 안일 수도 밖일 수도 있습니다.
- **Bonferroni, 3절처럼**: 위와 같은 판정을 3절 funnel처럼 그립니다. 점 높이를 자기 Order 수 기준 대비 비율로 바꿔서 Order 1개 기준 띠 하나로 판정이 보입니다. 환산한 높이라 세로 눈금에 숫자가 없습니다. √(2 ln M_N) 차트와 나란히 볼 때 씁니다.
- **√(2 ln M_N)**: 배수 = √(2 ln M_N), M_N = 같은 N(웨이퍼 수)에 찍힌 점 수(아래 칸). N마다 점 수가 달라 계단 모양이 됩니다. 점이 하나뿐인 N은 퍼짐을 알 수 없어 경계를 그리지 않고 판정하지 않습니다. 퍼짐 보정 없는 원래 식입니다.
- 비교를 위해 세 차트 모두 **Value만** 봅니다(bad를 합치지 않은 z). 3절 판정은 Value와 bad를 합친 결합 z라 경계 밖 수가 다를 수 있습니다.

In [ ]:
if len(res['combos']):
    plot_envelope('bonf')

In [ ]:
if len(res['combos']):
    plot_envelope('bonf1')

In [ ]:
if len(res['combos']):
    plot_envelope('memo')

## 부록 A. 데이터 점검
읽은 줄, 뺀 줄, 값이 둘 이상인 웨이퍼, 재작업 중복 등을 셉니다. STEP으로 묶인 모습과 Value 분포도 여기서 봅니다.

In [ ]:
display(quality)
print(f"STEP SEQ {so['step_seq'].nunique():,}개 → STEP {len(step_map):,}개 · STEP당 Order 최대 {step_map['Order 수'].max()}개 (Order가 많은 STEP부터)")
display(step_map.head(8).rename_axis('STEP'))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
cnt = np.bincount([len(s['seqs']) for s in steps])
axes[0].bar(np.arange(1, len(cnt)), cnt[1:], width=0.6, color=C['accent'])
axes[0].set(title='STEP별 Order 수', xlabel='Order 수', ylabel='STEP 수')
cnt = np.bincount([len(q['units']) for s in steps for q in s['seqs']])
axes[1].bar(np.arange(1, len(cnt)), cnt[1:], width=0.6, color=C['accent'])
axes[1].set(title='Order별 유닛 수', xlabel='유닛 수', ylabel='Order 수')
v, b, vn = data['value'], data['bad'], n_value.to_numpy(float)
if np.isfinite(b).any():
    bins = np.linspace(min(v.min(), vn.min(initial=np.inf)), max(v.max(), vn.max(initial=-np.inf)), 50)
    parts_ = [(v[b == 0], C['muted'], 'G'), (v[b == 1], C['bad'], 'B')] + ([(vn, C['axis'], 'N (계산 제외)')] if len(vn) else [])
    axes[2].hist([p[0] for p in parts_], bins=bins, stacked=True, color=[p[1] for p in parts_], label=[p[2] for p in parts_],
                 edgecolor=C['surface'], linewidth=0.5)
    axes[2].legend()
else:
    axes[2].hist(v, bins=50, color=C['muted'])
axes[2].set(title='웨이퍼 Value 분포', xlabel='Value', ylabel='웨이퍼 수')
for ax in axes:
    ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

## 부록 B. 판정 계산 값
자동 보정의 근거와 기준선 값입니다.

In [ ]:
print(f"Value 왜도 {skew:.2f} → " + {'rank': '순위 점수로 비교', 'log': '로그로 비교'}.get(tf, '값 그대로 비교'))
if part_table is not None:
    display(part_table)
    print('\n'.join(part_lines))
if data['B'] is not None:
    print(f"Value z와 bad z의 상관 r = {res['r']:.2f} → 결합 z = (z_Value + z_bad) ÷ √(2 + 2r)")
print(f"조합 {len(res['combos']):,}개 계산 (Order {'/'.join(map(str, range(1, MAX_DEPTH + 1)))}개: {' / '.join(f'{v:,}' for v in res['depth_count'][1:])}) · "
      f"기준선 보정에 쓴 조합 수 {' / '.join(f'{v:,}' for v in res['search_space'][1:])} "
      f"(5%를 Order 수 {sum(1 for m in res['search_space'][1:] if m)}개로 나눔) · {analyze_sec:.1f}초")
print(f"Order 1개 조합 z의 퍼짐 {res['spread_measured']:.2f}배 (우연이면 1) → "
      + (f"기준 z를 {res['spread']:.2f}배로 넓힘" if res['spread'] > 1 else '넓히지 않음' + ('' if SPREAD_ADJUST else ' (SPREAD_ADJUST = False)')))
display(pd.DataFrame({'기준 z': zk[1:]}, index=pd.Index(range(1, MAX_DEPTH + 1), name='Order 수')))
print(f"기준선 밖 {summary['over']:,}개 → 불량 판정 {summary['bad']}개 · 혐의 대상에 포함 {summary['inherited']}개 · 더 좁은 조합이 원인 {summary['explained']}개")

## 부록 C. 판정 방식 자세히
**조합**: 한 STEP 안에서 Order 1~`MAX_DEPTH`개를 골라, 고른 Order마다 정해진 유닛을 모두 지난 웨이퍼 묶음입니다. Order는 이웃하지 않아도 됩니다(`O3 → O17`). 웨이퍼 `MIN_N`장 이상만 보고, Order 3개 이상 조합은 Order 하나를 뺀 조합 중 하나가 z 1.96을 넘을 때만 계산합니다(계산량을 줄이려고).

**z (유의차)**: 조합이 나머지 웨이퍼보다 얼마나 높은지를, 우연히 흔들리는 폭(표준오차)의 몇 배인지로 나타낸 값입니다. Value와 bad 비율 각각 구해 하나로 합칩니다.
- 웨이퍼(`root_lot_id` + `wafer_id`) 한 장을 하나의 표본으로 봅니다. 표준편차는 전체 웨이퍼의 표준편차이고, 웨이퍼가 많을수록 표준오차가 작아져 funnel 모양이 됩니다.
- z = (조합 평균 − 전체 평균) ÷ (σ/√N) ÷ √(1 − N/전체 웨이퍼 수). 전체 평균에는 그 조합의 웨이퍼도 들어 있어서, 마지막 항이 없으면 큰 조합일수록 차이가 작게 나옵니다(예: 전체 1,130장 중 400장인 조합은 z가 1.24배). 조합을 나머지 웨이퍼와 비교하는 것과 같습니다.
- bad 비율 z는 bad가 드물고 웨이퍼가 적으면 정규 근사가 부풀려지므로 정확한 이항 꼬리확률로 맞춥니다(예: 25장 중 bad 13장, 전체 bad 16%일 때 4.96 → 4.01).
- **결합 z = (z_Value + z_bad) ÷ √(2 + 2r)**. r은 두 z가 우연히 같이 움직이는 정도(Order 1개 조합에서 잰 상관)로, 같은 정보를 두 번 세지 않게 합니다. 둘 다 나쁘면 크게 오르고 한쪽만 나쁘면 덜 오릅니다. good_bad가 없으면 Value z만 씁니다.

**기준선**: 조합을 수십만 개 보므로 기준을 올립니다(Bonferroni). 5%를 Order 수(1~`MAX_DEPTH`개 중 조합이 있는 것)로 나누고, Order 수마다 그 조합 수만큼 다시 나눕니다. 그래서 차트 전체에서 우연히 하나라도 넘을 확률이 5%입니다. `MAX_DEPTH`를 늘리면 Order 1개 조합까지 모든 기준이 조금씩 올라갑니다.

**퍼짐 보정**: Order 1개 조합(유닛 하나)의 결합 z는 우연이면 퍼짐이 1이어야 합니다. 같은 랏 웨이퍼가 함께 한 설비를 지나거나 시간에 따라 값이 변하면 실제로는 더 넓게 퍼집니다. 이 퍼짐(가운데 50% 폭 ÷ 1.349)이 1보다 크면 그 배수만큼 기준을 넓혀, 이 데이터에서 흔한 차이보다 뚜렷하게 튀는 조합만 남깁니다. 이론대로 퍼져 있으면 기준은 그대로이고, `SPREAD_ADJUST = False`로 끌 수 있습니다. **확실도** = 결합 z ÷ 기준 z입니다. funnel은 Order 수가 다른 조합도 같은 기준선으로 보이게 점의 높이를 맞춰 그리므로, 세로 눈금에는 값을 적지 않습니다.

**good_bad N**: N인 웨이퍼는 판정과 표 계산에서 빼고, Value 분포 차트(4절 '이 경로 vs 나머지'의 빈 원, 부록 A)에만 그립니다. G · B가 하나도 없으면 Value만으로 판정하고 이때는 모든 웨이퍼를 씁니다.

**자동 보정**: Value가 한쪽으로 길게 치우치면(왜도 1 초과) 순위 점수(순위를 정규분포 점수로 옮긴 값)로 바꿔 비교합니다. part마다 Value나 bad 비율 수준이 뚜렷하게 다르면 part 평균을 맞춥니다. 표 · 차트에는 원래 값을 보여 줍니다. Value가 클수록 좋은 값(`HIGHER_IS_WORSE = False`)이면 부호를 뒤집어 판정하고, funnel 세로축도 뒤집어 낮은 조합이 아래에 오게 그립니다.

**정리 규칙**
- Order 2개 이상 조합은 Order 하나를 뺀 조합의 나머지 웨이퍼보다도 기준선만큼 나빠야 불량입니다. 아니면 '혐의 대상에 포함'입니다(불량 유닛 하나가 그 유닛을 포함한 조합을 모두 끌어올리기 때문).
- 불량 조합이 더 좁은 불량 조합의 웨이퍼를 빼면 기준선 안으로 들어오면 '더 좁은 조합이 원인'입니다.
- 기준선 밖 점마다 같은 STEP에서 유닛을 공유하는 불량 조합 중 **초과 bad**(웨이퍼 × (bad 비율 − 전체 bad 비율))가 가장 큰 것을 혐의 대상으로 묶고, 초과 bad 순으로 순위를 매깁니다.
- 웨이퍼가 `SAME_WAFERS`(90%) 이상 같은 혐의 대상은 데이터로 가를 수 없어 한 줄로 묶습니다(비고의 '같은 웨이퍼').
- 다른 STEP의 혐의 대상과 웨이퍼가 겹칠 때, 상대의 웨이퍼를 빼면 효과가 사라지고 상대는 남으면 '다른 STEP 탓 의심'입니다.